# Physics-Grounded Synthetic Supervision for Controlled CT Restoration

---

## Research Map — This notebook argues for a **method**, supported by completed evidence blocks

### Core problem
In medical imaging, perfectly paired training data are rare. We usually **do not** have the same patient scanned once in a pristine state and once again under a realistic degradation process with identical anatomy.

So the scientific question here is not simply:

> “Can I build a stronger restoration network?”

It is:

> **When true paired data are unavailable, can physics-grounded degradation synthesis provide supervision that is reliable enough to train a clinically relevant and conservative CT restoration model?**

---

## What is the main contribution here?
This notebook uses **V-Ultimate** as the implementation vehicle, but the broader contribution is the **methodology**:

1. **Generate paired supervision from physics-grounded degradations** rather than arbitrary image corruption.
2. **Train a restoration policy under controlled, do-no-harm constraints** so edits remain conservative rather than aggressive.
3. **Judge the synthetic supervision through transfer, clinical-proxy behavior, and anatomy-aware evaluation**, not by visual sharpness alone.

So the scientific claim is **not** that this exact architecture is the final or universally best restoration system.

The claim is narrower and more useful:

> **Physics-grounded synthetic supervision can be a credible way to train CT restoration when real paired data are missing, provided that the learned behavior is clinically useful, externally transferable, and anatomically structured.**

---

## What this notebook actually shows in this run
The completed results support a **three-part methodology story**:

- On a **strict held-out RSNA OOD CT/CTA set (N = 50)**, V-Ultimate achieved **mean target gain = 0.0324**, **target win rate = 66%**, **iatrogenic rate = 32%**, and the **highest PSNR = 42.74 dB** among the compared methods.
- On **external paired Mayo low-dose CT (10 unique matched series)**, V-Ultimate improved over Quarter Dose by **+2.35 dB PSNR**, **+0.00203 MAE reduction**, **+0.04099 SSIM**, and **100% PSNR / MAE / SSIM win rate**, with a bootstrap **95% CI of +2.03 to +2.67 dB** for PSNR.
- In the same Mayo transfer test, V-Ultimate remained **more conservative than Gaussian smoothing**, with **mean absolute change vs Quarter = 0.00316** and **max absolute change = 0.02463**, compared with Gaussian’s **0.00690** mean and **0.15326** max.
- In **TotalSegmentator analysis on the same 10 Mayo series**, the original threshold of **0.05** revealed essentially no changed voxels, showing that the model’s edits are very small. After lowering the analysis threshold to **0.01**, the edits became measurable and structured: mean Dice vs Full improved from **0.6137 (Quarter)** to **0.7395 (V-Ultimate)** over **238 masks**, with changes concentrating in specific anatomical structures rather than uniformly across the volume.

These results do **not** say that V-Ultimate wins every metric against every baseline.  
Instead, they show that the synthetic-supervision pipeline produces a model with a strong **balance of usefulness, restraint, external transfer, and anatomical structure**, rather than a narrow one-metric win.

---

## Evidence chain used in this notebook

| Evidence block | What this run shows | Why it matters for the methodology |
|---|---|---|
| **Clinical Rescue Matrix (strict RSNA OOD, N = 50)** | V-Ultimate achieves the **highest overall mean target gain (0.0324)**, the **highest target win rate (66%)**, and the **highest PSNR (42.74 dB)**, while keeping iatrogenic rate tied for the lowest among the stronger methods | Shows that the synthetic supervision is **internally useful**, not just visually plausible |
| **Mayo external paired transfer (10 unique matched series)** | V-Ultimate achieves **+2.35 dB PSNR**, **+0.04099 SSIM**, and **100% PSNR win rate** over Quarter Dose, with consistently positive bootstrap confidence intervals | This is the strongest **sim-to-real transfer** evidence for the synthetic supervision claim |
| **TotalSegmentator anatomy analysis (10 Mayo series)** | At a coarse threshold the edits are too small to register; at a finer threshold they become visible and are **anatomically structured**, while Dice improves from **0.6137 to 0.7395** | Shows that the restoration is **conservative but not random**: edits are small, structured, and anatomically meaningful |

---

## What this notebook does **not** claim
To stay scientifically honest, this notebook does **not** prove that synthetic low-quality CT is distributionally identical to every real scanner protocol.

It also does **not** claim that V-Ultimate is the best possible filter on every metric or every task.

Instead, it argues something narrower and stronger:

> If a model trained on physics-grounded synthetic pairs repeatedly shows clinically relevant behavior on strict OOD data, transfers to real external low-dose CT, and makes small but anatomically structured edits, then the synthetic supervision is **useful and empirically supported as a training signal**, even if it is not a perfect replica of reality.

---



# ═══════════════════════════════════════════════════════════════
# Method Setup — Synthetic supervision + controlled restoration
# ═══════════════════════════════════════════════════════════════

This section keeps the **original training code and model implementation intact**.  
The purpose of the rewritten narrative is to make the scientific framing clearer:

- **Primary contribution:** a way to generate and validate training supervision when real pairs are absent
- **Implementation vehicle:** V-Ultimate (the restoration model used to test that supervision)
- **Evaluation principle:** safety and transfer matter more than “sharpest looking image”

In other words, the model is important, but it serves the larger question:

> **Can physics-grounded synthetic pairs support a restoration system that generalizes beyond the simulator?**


In [3]:
# =====================================================================
# V-Ultimate FULL TRAINING CELL 
# =====================================================================

import os, gc, math, time, random
import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from contextlib import nullcontext
import torch.fft

# -----------------------------
# Environment
# -----------------------------
try:
    cv2.setNumThreads(0)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

USE_AMP = (device.type == "cuda")
AMP_CTX = lambda: torch.amp.autocast("cuda") if USE_AMP else nullcontext()
scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

torch.backends.cudnn.benchmark = True
if device.type == "cuda":
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# ============================================================
# Configuration
# ============================================================

# --- Data paths ---
RSNA_DATA_ROOT = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/series"
TRAIN_LOCALIZERS_CSV = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train_localizers.csv"
META_CSV = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv"

OUT_TRAIN_UIDS = "/kaggle/working/train_uids_ultimate.csv"
OUT_VAL_UIDS   = "/kaggle/working/val_uids_ultimate.csv"

SAVE_BEST = "/kaggle/working/deblur_ultimate_film_auth_best.pt"
SAVE_LAST = "/kaggle/working/deblur_ultimate_film_auth_last.pt"

# --- Training scale ---
SEED = 2026
N_TRAIN_UIDS = 300
N_VAL_UIDS   = 20
EPOCHS = 24
BATCH_SIZE = 32
BATCHES_PER_EPOCH = 400
NUM_WORKERS = 4

# --- Optimiser ---
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

# --- Image parameters ---
TARGET_D, TARGET_H, TARGET_W = 64, 448, 448
PATCH_SIZE = 112
PATCHES_PER_SLICE = 1

HU_MIN, HU_MAX = -1024.0, 3072.0
HU_RANGE = HU_MAX - HU_MIN

# --- Degradation parameters ---
DIFFUSION_ALPHA = 0.20
BLUR_LEVELS = [0, 1, 3, 5, 8]
BLUR_LEVEL_MAX = float(max(BLUR_LEVELS))
BLUR_T_MAX = BLUR_LEVEL_MAX

P_IDENTITY = 0.20
ENABLE_MOTION = True
P_MOTION = 0.15

PEAK_RANGE_QUARTER, SIGMA_E_QUARTER = (3000.0, 6000.0), (0.01, 0.02)
PEAK_RANGE_EXTREME, SIGMA_E_EXTREME = (1000.0, 3000.0), (0.02, 0.04)

REGIME_PROBS = {"clean": 0.30, "typical": 0.45, "hard": 0.15, "motion": 0.10}

# --- Anatomy-aware patch sampling ---
ANATOMY_REJECT_TRIES = 15
ANATOMY_MEAN_TH = 0.05
ANATOMY_STD_TH  = 0.02
P_RANDOM_PATCH  = 0.10

# --- Loss weights ---
W_CHARBONNIER = 1.0
W_SSIM = 0.20
W_SOBEL = 0.10
W_LAP = 0.05
W_FFT = 0.05
FFT_FCUTOFF = 0.20
FFT_ONLY_IF_T_LE = 10.0

W_CHANGE_ID = 10.0
W_LOW_T_EDIT = 2.0
LOW_T_THR = 0.20
W_AUTH_TV = 1e-3

RES_MIN, RES_MAX = 0.02, 0.15

# ============================================================
# Random seed
# ============================================================

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

def _normalize_probs(d):
    s = float(sum(d.values()))
    return {k: float(v) / s for k, v in d.items()}

REGIME_PROBS = _normalize_probs(REGIME_PROBS)

# ============================================================
# UID list construction
# ============================================================

def build_ct_only_uid_lists(meta_csv, rsna_series_root, localizers_csv, n_train, n_val, seed):
    rsna_uids = set(
        [u for u in os.listdir(rsna_series_root)
         if os.path.isdir(os.path.join(rsna_series_root, u)) and not u.startswith(".")]
    )

    localizer_uids = set()
    if localizers_csv and os.path.exists(localizers_csv):
        df_loc = pd.read_csv(localizers_csv)
        localizer_uids = set(df_loc[df_loc.columns[0]].astype(str).tolist())

    meta = pd.read_csv(meta_csv)
    meta_ct = meta[meta["Modality"].isin({"CT", "CTA"})]

    ct_candidates = [
        u for u in set(meta_ct["SeriesInstanceUID"].astype(str))
        if u in rsna_uids and u not in localizer_uids
    ]

    rng = random.Random(seed)
    rng.shuffle(ct_candidates)

    train_uids = ct_candidates[:min(n_train, len(ct_candidates))]
    rem_ct = [u for u in ct_candidates if u not in set(train_uids)]
    rng.shuffle(rem_ct)
    val_uids = rem_ct[:min(n_val, len(rem_ct))]
    return train_uids, val_uids

# ============================================================
# DICOM loading
# ============================================================

def get_sorted_dicom_files(series_path):
    files = [f for f in os.listdir(series_path) if not f.startswith(".")]
    pairs, ok = [], True
    for f in files:
        try:
            ds = pydicom.dcmread(os.path.join(series_path, f), stop_before_pixels=True, force=True)
            if getattr(ds, "InstanceNumber", None) is None:
                ok = False
                break
            pairs.append((int(ds.InstanceNumber), os.path.join(series_path, f)))
        except Exception:
            ok = False
            break
    if ok and len(pairs) == len(files):
        return [p[1] for p in sorted(pairs, key=lambda x: x[0])]
    return [os.path.join(series_path, f) for f in sorted(files)]

def load_series_volume(uid, series_root, target_shape=(64, 448, 448)):
    series_path = os.path.join(series_root, uid)
    if not os.path.isdir(series_path):
        return None

    dcm_files = get_sorted_dicom_files(series_path)
    tD, tH, tW = target_shape
    if len(dcm_files) < 10:
        return None

    if len(dcm_files) != tD:
        idxs = np.linspace(0, len(dcm_files) - 1, tD).astype(int)
        dcm_files = [dcm_files[i] for i in idxs]

    slices = []
    for fp in dcm_files:
        try:
            ds = pydicom.dcmread(fp, force=True)
            hu = ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1.0)) \
                 + float(getattr(ds, "RescaleIntercept", 0.0))
            x = (np.clip(hu, HU_MIN, HU_MAX) - HU_MIN) / HU_RANGE
            x = cv2.resize(x, (tW, tH), interpolation=cv2.INTER_LINEAR)
            slices.append(x.astype(np.float32))
        except Exception:
            continue

    if len(slices) < int(0.8 * tD):
        return None

    while len(slices) < tD:
        slices.append(slices[-1].copy())

    return np.stack(slices[:tD], axis=0).astype(np.float32)

class VolumeLRU:
    def __init__(self, max_items=12):
        self.max_items = int(max_items)
        self.cache = {}
        self.order = []

    def get(self, key):
        if key not in self.cache:
            return None
        self.order.remove(key)
        self.order.append(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.order.remove(key)
        self.cache[key] = value
        self.order.append(key)
        if len(self.order) > self.max_items:
            old = self.order.pop(0)
            self.cache.pop(old, None)

# ============================================================
# Degradation engine
# ============================================================

def gaussian_psf_surrogate(img01, blur_level, alpha=0.20):
    if blur_level <= 0:
        return img01
    sigma = math.sqrt(max(1e-8, 2.0 * alpha * float(blur_level)))
    return np.clip(
        cv2.GaussianBlur(img01, (0, 0), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REPLICATE),
        0.0, 1.0
    )

def motion_artifact_surrogate(img01, length=None, angle=None):
    length = length or random.choice([3, 5, 7, 9, 11])
    if length <= 1:
        return img01
    angle = angle or random.uniform(0, 180)

    k = np.zeros((length, length), dtype=np.float32)
    c = length // 2
    cos_a, sin_a = np.cos(np.radians(angle)), np.sin(np.radians(angle))
    for i in range(length):
        x = int(c + (i - c) * cos_a)
        y = int(c + (i - c) * sin_a)
        if 0 <= x < length and 0 <= y < length:
            k[y, x] = 1.0
    if k.sum() > 0:
        k /= k.sum()

    return np.clip(cv2.filter2D(img01, -1, k, borderType=cv2.BORDER_REPLICATE), 0.0, 1.0)

def mixed_poisson_gaussian(img01, mode="quarter"):
    if mode == "clean":
        return img01
    peak_rng, sigma_rng = (
        (PEAK_RANGE_EXTREME, SIGMA_E_EXTREME) if mode == "extreme"
        else (PEAK_RANGE_QUARTER, SIGMA_E_QUARTER)
    )
    peak = random.uniform(*peak_rng)
    sigma_e = random.uniform(*sigma_rng)
    noisy_p = np.random.poisson(np.clip(img01 * peak, 0, None)).astype(np.float32) / peak
    noisy_g = np.random.randn(*img01.shape).astype(np.float32) * sigma_e
    return np.clip(noisy_p + noisy_g, 0.0, 1.0)

def _choice_weighted(prob_dict):
    r = random.random()
    acc = 0.0
    for k, p in prob_dict.items():
        acc += p
        if r <= acc:
            return k
    return list(prob_dict.keys())[-1]

def _sample_regime_params():
    reg = _choice_weighted(REGIME_PROBS)
    if reg == "clean":
        return 0, "clean", False
    if reg == "typical":
        return random.choice([1, 3, 5]), "quarter", False
    if reg == "hard":
        return random.choice([3, 5, 8]), "extreme", False
    if reg == "motion":
        return random.choice([1, 3, 5]), random.choice(["quarter", "extreme"]), True
    return 3, "quarter", False

# ============================================================
# Dataset
# ============================================================

class CTDeblur25D(Dataset):
    def __init__(self, uids, series_root, target_shape=(64, 448, 448), patch_size=112, patches_per_slice=1):
        self.uids = list(uids)
        self.series_root = series_root
        self.target_shape = target_shape
        self.patch_size = int(patch_size)
        self.patches_per_slice = int(patches_per_slice)
        self.cache = VolumeLRU(max_items=8)
        self.items = [(ui, z) for ui in range(len(self.uids)) for z in range(1, target_shape[0] - 1)]

    def __len__(self):
        return len(self.items)

    def _sample_patch_xy(self, cent, ps):
        H, W = cent.shape
        if random.random() < P_RANDOM_PATCH:
            return np.random.randint(0, H - ps + 1), np.random.randint(0, W - ps + 1)

        for _ in range(ANATOMY_REJECT_TRIES):
            y = np.random.randint(0, H - ps + 1)
            x = np.random.randint(0, W - ps + 1)
            patch = cent[y:y+ps, x:x+ps]
            if patch.mean() > ANATOMY_MEAN_TH and patch.std() > ANATOMY_STD_TH:
                return y, x

        return (H - ps) // 2, (W - ps) // 2

    def __getitem__(self, idx):
        ui, z = self.items[idx]
        uid = self.uids[ui]

        vol = self.cache.get(uid)
        if vol is None:
            vol = load_series_volume(uid, self.series_root, self.target_shape)
            if vol is None:
                return self.__getitem__(random.randint(0, len(self.items) - 1))
            self.cache.put(uid, vol)

        ps = self.patch_size
        y, x = self._sample_patch_xy(vol[z], ps)

        clean = vol[z][y:y+ps, x:x+ps].copy()
        pp = vol[z-1][y:y+ps, x:x+ps].copy()
        cc = vol[z][y:y+ps, x:x+ps].copy()
        nn_ = vol[z+1][y:y+ps, x:x+ps].copy()

        is_identity = 0.0
        if random.random() < P_IDENTITY:
            blur_level, dose_mode, do_motion, is_identity = 0, "clean", False, 1.0
        else:
            blur_level, dose_mode, do_motion = _sample_regime_params()
            do_motion = do_motion and ENABLE_MOTION and (random.random() < P_MOTION)

        bp = gaussian_psf_surrogate(pp, blur_level, alpha=DIFFUSION_ALPHA)
        bc = gaussian_psf_surrogate(cc, blur_level, alpha=DIFFUSION_ALPHA)
        bn = gaussian_psf_surrogate(nn_, blur_level, alpha=DIFFUSION_ALPHA)

        if do_motion:
            L = random.choice([3, 5, 7, 9, 11])
            A = random.uniform(0, 180)
            bp = motion_artifact_surrogate(bp, L, A)
            bc = motion_artifact_surrogate(bc, L, A)
            bn = motion_artifact_surrogate(bn, L, A)

        if dose_mode != "clean":
            bp = mixed_poisson_gaussian(bp, dose_mode)
            bc = mixed_poisson_gaussian(bc, dose_mode)
            bn = mixed_poisson_gaussian(bn, dose_mode)

        cp = clean.copy()
        if random.random() > 0.5:
            cp, bp, bc, bn = cp[::-1].copy(), bp[::-1].copy(), bc[::-1].copy(), bn[::-1].copy()
        if random.random() > 0.5:
            cp, bp, bc, bn = cp[:, ::-1].copy(), bp[:, ::-1].copy(), bc[:, ::-1].copy(), bn[:, ::-1].copy()
        k = random.randint(0, 3)
        if k > 0:
            cp, bp, bc, bn = np.rot90(cp, k).copy(), np.rot90(bp, k).copy(), np.rot90(bc, k).copy(), np.rot90(bn, k).copy()

        t_norm = float(blur_level) / BLUR_LEVEL_MAX if BLUR_LEVEL_MAX > 0 else 0.0

        inp = np.stack([bp, bc, bn, np.full_like(bc, t_norm, dtype=np.float32)], axis=0).astype(np.float32)
        tgt = cp[np.newaxis, ...].astype(np.float32)

        meta = np.array([
            t_norm,
            1.0 if do_motion else 0.0,
            1.0 if dose_mode == "clean" else 0.0,
            is_identity
        ], dtype=np.float32)

        return (
            torch.from_numpy(inp).float(),
            torch.from_numpy(tgt).float(),
            torch.from_numpy(meta).float()
        )

class UIDBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, seed=42, batches_per_epoch=None):
        self.dataset = dataset
        self.batch_size = int(batch_size)
        self.rng = random.Random(seed)

        self.by_ui = {}
        for idx, (ui, z) in enumerate(dataset.items):
            self.by_ui.setdefault(ui, []).append(idx)

        self.ui_keys = list(self.by_ui.keys())
        self.batches_per_epoch = int(batches_per_epoch) if batches_per_epoch else len(dataset) // self.batch_size

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        for _ in range(self.batches_per_epoch):
            ui = self.rng.choice(self.ui_keys)
            pool = self.by_ui[ui]
            if len(pool) >= self.batch_size:
                yield self.rng.sample(pool, self.batch_size)
            else:
                yield [self.rng.choice(pool) for _ in range(self.batch_size)]

# ============================================================
# New model: FiLM + authority map
# ============================================================

class SEBlock(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, max(1, c // r), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(1, c // r), c, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x)

class ResBlockPhysics(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(oc, oc, 3, padding=1, bias=True),
        )
        self.shortcut = nn.Conv2d(ic, oc, 1, bias=True) if ic != oc else nn.Identity()

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x), inplace=True)

class UpsamplePhysicsUltimate(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2.0, mode="nearest"),
            nn.Conv2d(ic, oc, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.up(x)

class FiLM2d(nn.Module):
    def __init__(self, channels, meta_dim=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels * 2),
        )
        self.channels = channels

    def forward(self, x, meta):
        gb = self.net(meta)
        gamma, beta = torch.chunk(gb, 2, dim=1)
        gamma = gamma.view(-1, self.channels, 1, 1)
        beta = beta.view(-1, self.channels, 1, 1)
        return x * (1.0 + gamma) + beta

class DeblurUNet25D_Ultimate(nn.Module):
    def __init__(
        self,
        in_ch=4,
        out_ch=1,
        base=32,
        res_min=0.02,
        res_max=0.15,
        meta_dim=4,
        film_hidden=64,
        authority_bias_init=2.0,
    ):
        super().__init__()
        self.res_min = float(res_min)
        self.res_max = float(res_max)
        self.meta_dim = int(meta_dim)

        c = [base, base * 2, base * 4, base * 8]

        self.se = SEBlock(in_ch)

        self.enc1 = ResBlockPhysics(in_ch, c[0])
        self.enc2 = ResBlockPhysics(c[0], c[1])
        self.enc3 = ResBlockPhysics(c[1], c[2])
        self.enc4 = ResBlockPhysics(c[2], c[3])
        self.pool = nn.MaxPool2d(2)

        self.up3 = UpsamplePhysicsUltimate(c[3], c[2])
        self.dec3 = ResBlockPhysics(c[2] * 2, c[2])

        self.up2 = UpsamplePhysicsUltimate(c[2], c[1])
        self.dec2 = ResBlockPhysics(c[1] * 2, c[1])

        self.up1 = UpsamplePhysicsUltimate(c[1], c[0])
        self.dec1 = ResBlockPhysics(c[0] * 2, c[0])

        self.film_e1 = FiLM2d(c[0], meta_dim=meta_dim, hidden=film_hidden)
        self.film_e2 = FiLM2d(c[1], meta_dim=meta_dim, hidden=film_hidden)
        self.film_e3 = FiLM2d(c[2], meta_dim=meta_dim, hidden=film_hidden)
        self.film_e4 = FiLM2d(c[3], meta_dim=meta_dim, hidden=film_hidden)
        self.film_d3 = FiLM2d(c[2], meta_dim=meta_dim, hidden=film_hidden)
        self.film_d2 = FiLM2d(c[1], meta_dim=meta_dim, hidden=film_hidden)
        self.film_d1 = FiLM2d(c[0], meta_dim=meta_dim, hidden=film_hidden)

        self.out_conv = nn.Conv2d(c[0], out_ch, 1, bias=True)

        self.auth_head = nn.Sequential(
            nn.Conv2d(c[0], c[0] // 2, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(c[0] // 2, 1, 1, bias=True),
        )
        nn.init.constant_(self.auth_head[-1].bias, float(authority_bias_init))

    def forward(self, x, meta=None, return_aux=False):
        bc = x[:, 1:2]
        tch = x[:, 3:4]

        if meta is None:
            B = x.shape[0]
            t_scalar = torch.mean(tch, dim=(2, 3)).view(B, 1)
            zeros = torch.zeros(B, 3, device=x.device, dtype=x.dtype)
            meta = torch.cat([t_scalar, zeros], dim=1)

        e1 = self.enc1(self.se(x))
        e1 = self.film_e1(e1, meta)

        e2 = self.enc2(self.pool(e1))
        e2 = self.film_e2(e2, meta)

        e3 = self.enc3(self.pool(e2))
        e3 = self.film_e3(e3, meta)

        e4 = self.enc4(self.pool(e3))
        e4 = self.film_e4(e4, meta)

        d3 = self.dec3(torch.cat([self.up3(e4), e3], dim=1))
        d3 = self.film_d3(d3, meta)

        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d2 = self.film_d2(d2, meta)

        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        d1 = self.film_d1(d1, meta)

        residual = torch.tanh(self.out_conv(d1))

        base_rmax = self.res_min + (self.res_max - self.res_min) * tch
        authority = torch.sigmoid(self.auth_head(d1))
        rmax_map = base_rmax * authority

        pred_soft = (bc + residual * rmax_map).clamp(0.0, 1.0)
        pred = torch.where(tch <= 1e-8, bc, pred_soft)

        if return_aux:
            return pred, {
                "authority": authority,
                "rmax_map": rmax_map,
                "residual": residual,
            }
        return pred

# ============================================================
# Loss functions
# ============================================================

def charbonnier_loss(pred, target, eps=1e-3):
    return torch.mean(torch.sqrt((pred - target) ** 2 + eps ** 2))

def fft_spectrum_loss(pred, target, fft_mask):
    pred_fft = torch.fft.rfft2(pred.float(), dim=(-2, -1), norm="ortho")
    tgt_fft = torch.fft.rfft2(target.float(), dim=(-2, -1), norm="ortho")
    m = fft_mask.view(1, 1, *fft_mask.shape)
    return charbonnier_loss(torch.abs(pred_fft) * m, torch.abs(tgt_fft) * m)

def ssim_loss(pred, target, window_size=11):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    pad = window_size // 2

    mu_x = F.avg_pool2d(pred, window_size, stride=1, padding=pad)
    mu_y = F.avg_pool2d(target, window_size, stride=1, padding=pad)

    sigma_x2 = F.avg_pool2d(pred ** 2, window_size, stride=1, padding=pad) - mu_x ** 2
    sigma_y2 = F.avg_pool2d(target ** 2, window_size, stride=1, padding=pad) - mu_y ** 2
    sigma_xy = F.avg_pool2d(pred * target, window_size, stride=1, padding=pad) - mu_x * mu_y

    ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / (
        (mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x2 + sigma_y2 + C2) + 1e-8
    )
    return 1.0 - ssim_map.mean()

def _make_fft_mask(H, W, fcut=0.20, device="cpu"):
    fy = torch.fft.fftfreq(H, d=1.0, device=device).view(H, 1).abs()
    fx = torch.fft.rfftfreq(W, d=1.0, device=device).view(1, W // 2 + 1).abs()
    return (torch.sqrt(fx * fx + fy * fy) >= fcut).float()

class UltimatePhysicsLoss(nn.Module):
    def __init__(self, patch_size=112):
        super().__init__()
        self.sobel_x = torch.tensor([[[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]]).view(1, 1, 3, 3).to(device)
        self.sobel_y = torch.tensor([[[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]]).view(1, 1, 3, 3).to(device)
        self.lap = torch.tensor([[[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]]).view(1, 1, 3, 3).to(device)
        self.register_buffer("fft_mask", _make_fft_mask(patch_size, patch_size, fcut=FFT_FCUTOFF, device=device))

    def forward(self, pred, target, allow_fft=False):
        total = W_CHARBONNIER * charbonnier_loss(pred, target)
        total += W_SSIM * ssim_loss(pred, target)

        p_pad = F.pad(pred, (1, 1, 1, 1), mode="replicate")
        t_pad = F.pad(target, (1, 1, 1, 1), mode="replicate")

        total += W_SOBEL * (
            charbonnier_loss(F.conv2d(p_pad, self.sobel_x), F.conv2d(t_pad, self.sobel_x))
            + charbonnier_loss(F.conv2d(p_pad, self.sobel_y), F.conv2d(t_pad, self.sobel_y))
        )
        total += W_LAP * charbonnier_loss(F.conv2d(p_pad, self.lap), F.conv2d(t_pad, self.lap))

        if allow_fft:
            total += W_FFT * fft_spectrum_loss(pred, target, self.fft_mask)
        return total

def alg_humility_penalty(pred, inp, is_id):
    center = inp[:, 1:2]
    per_sample = torch.mean(torch.abs(pred - center), dim=(1, 2, 3))
    return torch.mean(per_sample * is_id * W_CHANGE_ID)

def low_t_edit_penalty(pred, inp, meta):
    center = inp[:, 1:2]
    t_norm = meta[:, 0]
    low_mask = (t_norm <= LOW_T_THR).float()
    per_sample = torch.mean(torch.abs(pred - center), dim=(1, 2, 3))
    return torch.mean(per_sample * low_mask * W_LOW_T_EDIT)

def authority_tv_penalty(authority):
    dy = torch.abs(authority[:, :, 1:, :] - authority[:, :, :-1, :]).mean()
    dx = torch.abs(authority[:, :, :, 1:] - authority[:, :, :, :-1]).mean()
    return (dx + dy) * W_AUTH_TV

# ============================================================
# Validation
# ============================================================

@torch.no_grad()
def eval_model_psnr(model, val_uids, series_root, target_shape=(64, 448, 448), max_uids=8):
    model.eval()
    scores = []
    cache = VolumeLRU(max_items=2)

    for uid in list(val_uids)[:max_uids]:
        vol = cache.get(uid)
        if vol is None:
            vol = load_series_volume(uid, series_root, target_shape)
            if vol is None:
                continue
            cache.put(uid, vol)

        D = vol.shape[0]
        for blur_level in [3, 8]:
            for z in range(1, D - 1, 8):
                cl = vol[z].astype(np.float32)
                prev = vol[z - 1].astype(np.float32)
                cent = vol[z].astype(np.float32)
                next_ = vol[z + 1].astype(np.float32)

                bp = mixed_poisson_gaussian(gaussian_psf_surrogate(prev, blur_level, DIFFUSION_ALPHA), "quarter")
                bc = mixed_poisson_gaussian(gaussian_psf_surrogate(cent, blur_level, DIFFUSION_ALPHA), "quarter")
                bn = mixed_poisson_gaussian(gaussian_psf_surrogate(next_, blur_level, DIFFUSION_ALPHA), "quarter")

                t_norm = float(blur_level) / BLUR_LEVEL_MAX
                inp_np = np.stack([bp, bc, bn, np.full_like(bc, t_norm)], axis=0).astype(np.float32)
                inp_t = torch.from_numpy(inp_np).unsqueeze(0).to(device)

                meta_np = np.array([[t_norm, 0.0, 0.0, 0.0]], dtype=np.float32)
                meta_t = torch.from_numpy(meta_np).to(device)

                with AMP_CTX():
                    pred = model(inp_t, meta=meta_t)[0, 0].float().cpu().numpy()

                mse = float(np.mean((pred - cl) ** 2))
                scores.append(99.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse))

    return float(np.mean(scores)) if scores else None


Device: cuda


# Scientific-Method Training Design for the Synthetic-Supervision Pipeline

## Why this training design matters scientifically
If a project claims that synthetic supervision is trustworthy, then the evaluation protocol itself must also be trustworthy.

A common failure mode in ML projects is:
1. train a model,
2. peek at the final test set,
3. change the method based on those failures,
4. report the same test set as “independent evidence.”

That weakens the credibility of any claimed generalization.

This notebook avoids that mistake by using a **firewalled training design**:

- a main training split,
- a separate refinement split,
- and an external evaluation stage kept conceptually distinct.

## Core idea in one sentence
> **Use internal data to improve the training process, but do not let the external evidence become part of the tuning loop.**

## Why this matters for the methodology claim
Because the project is not only proposing a model.  
It is proposing a **way to create supervision from simulated degradations** and then argue that the supervision is meaningful.

That argument only works if the external evidence remains genuinely external.

## What this section contributes to the final story
This training design supports the statement that later results—especially on Mayo—are being used as **validation of transfer**, not as hidden tuning feedback.


In [ ]:
# =====================================================================
# V-Ultimate scientific-method training cell
# Two-stage training:
#   Stage A: base training on TRAIN split
#   Stage B: failure-driven refinement on REFINE split only
#
# Goal:
#   improve the model using observed failure modes
#   without leaking information from the external test set
# =====================================================================

import os, gc, math, time, random, json
import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from contextlib import nullcontext
import torch.fft

# -----------------------------
# environment
# -----------------------------
try:
    cv2.setNumThreads(0)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

USE_AMP = (device.type == "cuda")
AMP_CTX = lambda: torch.amp.autocast("cuda") if USE_AMP else nullcontext()
scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

torch.backends.cudnn.benchmark = True
if device.type == "cuda":
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# -----------------------------
# require architecture
# -----------------------------
if "DeblurUNet25D_Ultimate" not in globals():
    raise RuntimeError("请先运行新版 architecture cell（FiLM + authority map）。")

# ============================================================
# config
# ============================================================

# --- paths ---
RSNA_DATA_ROOT = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/series"
TRAIN_LOCALIZERS_CSV = "/kaggle/input/rsna-intracranial-aneurysm-detection/train_localizers.csv"
META_CSV = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv"

OUTDIR = "/kaggle/working/vultimate_scientific"
os.makedirs(OUTDIR, exist_ok=True)

OUT_TRAIN_UIDS  = os.path.join(OUTDIR, "train_uids.csv")
OUT_REFINE_UIDS = os.path.join(OUTDIR, "refine_uids.csv")
OUT_VAL_UIDS    = os.path.join(OUTDIR, "val_uids.csv")
OUT_HARD_UIDS   = os.path.join(OUTDIR, "hard_uids_stageB.csv")

SAVE_STAGEA_BEST = os.path.join(OUTDIR, "deblur_stageA_best.pt")
SAVE_STAGEA_LAST = os.path.join(OUTDIR, "deblur_stageA_last.pt")
SAVE_STAGEB_BEST = os.path.join(OUTDIR, "deblur_stageB_best.pt")
SAVE_STAGEB_LAST = os.path.join(OUTDIR, "deblur_stageB_last.pt")

# --- dataset size ---
SEED = 2026
N_TRAIN_UIDS  = 300
N_REFINE_UIDS = 60
N_VAL_UIDS    = 20

# --- stage A training ---
EPOCHS_STAGE_A = 18
BATCHES_PER_EPOCH_A = 300

# --- stage B refinement ---
EPOCHS_STAGE_B = 8
BATCHES_PER_EPOCH_B = 220
N_HARD_UIDS = 24           # 从 refine 集里挑多少个 hardest cases
HARD_UID_REPEAT = 3        # Stage B 中对 hard UID 做重复强化

# --- optimizer ---
BATCH_SIZE = 32
NUM_WORKERS = 4
LR_STAGE_A = 2e-4
LR_STAGE_B = 8e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

# --- image ---
TARGET_D, TARGET_H, TARGET_W = 64, 448, 448
PATCH_SIZE = 112
PATCHES_PER_SLICE = 1

HU_MIN, HU_MAX = -1024.0, 3072.0
HU_RANGE = HU_MAX - HU_MIN

# --- degradation ---
DIFFUSION_ALPHA = 0.20
BLUR_LEVELS = [0, 1, 3, 5, 8]
BLUR_LEVEL_MAX = float(max(BLUR_LEVELS))
BLUR_T_MAX = BLUR_LEVEL_MAX

P_IDENTITY = 0.20
ENABLE_MOTION = True
P_MOTION = 0.15

PEAK_RANGE_QUARTER, SIGMA_E_QUARTER = (3000.0, 6000.0), (0.01, 0.02)
PEAK_RANGE_EXTREME, SIGMA_E_EXTREME = (1000.0, 3000.0), (0.02, 0.04)

REGIME_PROBS_STAGE_A = {"clean": 0.30, "typical": 0.45, "hard": 0.15, "motion": 0.10}
REGIME_PROBS_STAGE_B = {"clean": 0.20, "typical": 0.35, "hard": 0.30, "motion": 0.15}

# --- anatomy-aware crop ---
ANATOMY_REJECT_TRIES = 15
ANATOMY_MEAN_TH = 0.05
ANATOMY_STD_TH  = 0.02
P_RANDOM_PATCH  = 0.10

# --- loss weights ---
W_CHARBONNIER = 1.0
W_SSIM = 0.20
W_SOBEL = 0.10
W_LAP = 0.05
W_FFT = 0.05
FFT_FCUTOFF = 0.20
FFT_ONLY_IF_T_LE = 10.0

W_CHANGE_ID = 10.0
W_LOW_T_EDIT = 2.0
LOW_T_THR = 0.20
W_AUTH_TV = 1e-3

# stage B 强一点，专门压过修复
W_LOW_T_EDIT_STAGE_B = 3.0
W_AUTH_TV_STAGE_B = 2e-3

RES_MIN, RES_MAX = 0.02, 0.15

# --- hard-case mining ---
MINE_EVAL_MAX_UIDS = None        # None = all refine uids
MINE_STRIDE_Z = 8
MINE_BLUR_LEVELS = [3, 8]
HARD_SCORE_W_PSNR = 1.0
HARD_SCORE_W_OVEREDIT = 2.0

# ============================================================
# seed
# ============================================================

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)

def _normalize_probs(d):
    s = float(sum(d.values()))
    return {k: float(v) / s for k, v in d.items()}

REGIME_PROBS_STAGE_A = _normalize_probs(REGIME_PROBS_STAGE_A)
REGIME_PROBS_STAGE_B = _normalize_probs(REGIME_PROBS_STAGE_B)

# ============================================================
# split logic
# scientific-method version:
#   train      -> stage A base learning
#   refine     -> failure analysis + stage B targeted improvement
#   val        -> held-out internal validation only
# external Mayo is untouched
# ============================================================

def build_ct_uid_lists_3way(meta_csv, rsna_series_root, localizers_csv, n_train, n_refine, n_val, seed):
    rsna_uids = set(
        [u for u in os.listdir(rsna_series_root)
         if os.path.isdir(os.path.join(rsna_series_root, u)) and not u.startswith(".")]
    )

    localizer_uids = set()
    if localizers_csv and os.path.exists(localizers_csv):
        df_loc = pd.read_csv(localizers_csv)
        localizer_uids = set(df_loc[df_loc.columns[0]].astype(str).tolist())

    meta = pd.read_csv(meta_csv)
    meta_ct = meta[meta["Modality"].isin({"CT", "CTA"})]

    ct_candidates = [
        u for u in set(meta_ct["SeriesInstanceUID"].astype(str))
        if u in rsna_uids and u not in localizer_uids
    ]

    rng = random.Random(seed)
    rng.shuffle(ct_candidates)

    total_need = n_train + n_refine + n_val
    picked = ct_candidates[:min(total_need, len(ct_candidates))]

    train_uids = picked[:min(n_train, len(picked))]
    rem = picked[len(train_uids):]

    refine_uids = rem[:min(n_refine, len(rem))]
    rem2 = rem[len(refine_uids):]

    val_uids = rem2[:min(n_val, len(rem2))]

    return train_uids, refine_uids, val_uids

# ============================================================
# DICOM loading
# ============================================================

def get_sorted_dicom_files(series_path):
    files = [f for f in os.listdir(series_path) if not f.startswith(".")]
    pairs, ok = [], True
    for f in files:
        try:
            ds = pydicom.dcmread(os.path.join(series_path, f), stop_before_pixels=True, force=True)
            if getattr(ds, "InstanceNumber", None) is None:
                ok = False
                break
            pairs.append((int(ds.InstanceNumber), os.path.join(series_path, f)))
        except Exception:
            ok = False
            break

    if ok and len(pairs) == len(files):
        return [p[1] for p in sorted(pairs, key=lambda x: x[0])]

    return [os.path.join(series_path, f) for f in sorted(files)]

def load_series_volume(uid, series_root, target_shape=(64, 448, 448)):
    series_path = os.path.join(series_root, uid)
    if not os.path.isdir(series_path):
        return None

    dcm_files = get_sorted_dicom_files(series_path)
    tD, tH, tW = target_shape
    if len(dcm_files) < 10:
        return None

    if len(dcm_files) != tD:
        idxs = np.linspace(0, len(dcm_files) - 1, tD).astype(int)
        dcm_files = [dcm_files[i] for i in idxs]

    slices = []
    for fp in dcm_files:
        try:
            ds = pydicom.dcmread(fp, force=True)
            hu = ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1.0)) \
                 + float(getattr(ds, "RescaleIntercept", 0.0))
            x = (np.clip(hu, HU_MIN, HU_MAX) - HU_MIN) / HU_RANGE
            x = cv2.resize(x, (tW, tH), interpolation=cv2.INTER_LINEAR)
            slices.append(x.astype(np.float32))
        except Exception:
            continue

    if len(slices) < int(0.8 * tD):
        return None

    while len(slices) < tD:
        slices.append(slices[-1].copy())

    return np.stack(slices[:tD], axis=0).astype(np.float32)

class VolumeLRU:
    def __init__(self, max_items=12):
        self.max_items = int(max_items)
        self.cache = {}
        self.order = []

    def get(self, key):
        if key not in self.cache:
            return None
        self.order.remove(key)
        self.order.append(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.order.remove(key)
        self.cache[key] = value
        self.order.append(key)
        if len(self.order) > self.max_items:
            old = self.order.pop(0)
            self.cache.pop(old, None)

# ============================================================
# physics degradation
# ============================================================

def gaussian_psf_surrogate(img01, blur_level, alpha=0.20):
    if blur_level <= 0:
        return img01
    sigma = math.sqrt(max(1e-8, 2.0 * alpha * float(blur_level)))
    return np.clip(
        cv2.GaussianBlur(img01, (0, 0), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REPLICATE),
        0.0, 1.0
    )

def motion_artifact_surrogate(img01, length=None, angle=None):
    length = length or random.choice([3, 5, 7, 9, 11])
    if length <= 1:
        return img01
    angle = angle or random.uniform(0, 180)

    k = np.zeros((length, length), dtype=np.float32)
    c = length // 2
    cos_a, sin_a = np.cos(np.radians(angle)), np.sin(np.radians(angle))
    for i in range(length):
        x = int(c + (i - c) * cos_a)
        y = int(c + (i - c) * sin_a)
        if 0 <= x < length and 0 <= y < length:
            k[y, x] = 1.0
    if k.sum() > 0:
        k /= k.sum()

    return np.clip(cv2.filter2D(img01, -1, k, borderType=cv2.BORDER_REPLICATE), 0.0, 1.0)

def mixed_poisson_gaussian(img01, mode="quarter"):
    if mode == "clean":
        return img01

    peak_rng, sigma_rng = (
        (PEAK_RANGE_EXTREME, SIGMA_E_EXTREME) if mode == "extreme"
        else (PEAK_RANGE_QUARTER, SIGMA_E_QUARTER)
    )
    peak = random.uniform(*peak_rng)
    sigma_e = random.uniform(*sigma_rng)
    noisy_p = np.random.poisson(np.clip(img01 * peak, 0, None)).astype(np.float32) / peak
    noisy_g = np.random.randn(*img01.shape).astype(np.float32) * sigma_e
    return np.clip(noisy_p + noisy_g, 0.0, 1.0)

def _choice_weighted(prob_dict):
    r = random.random()
    acc = 0.0
    for k, p in prob_dict.items():
        acc += p
        if r <= acc:
            return k
    return list(prob_dict.keys())[-1]

def _sample_regime_params(prob_dict):
    reg = _choice_weighted(prob_dict)
    if reg == "clean":
        return 0, "clean", False
    if reg == "typical":
        return random.choice([1, 3, 5]), "quarter", False
    if reg == "hard":
        return random.choice([3, 5, 8]), "extreme", False
    if reg == "motion":
        return random.choice([1, 3, 5]), random.choice(["quarter", "extreme"]), True
    return 3, "quarter", False

# ============================================================
# datasets
# ============================================================

class CTDeblur25D(Dataset):
    def __init__(
        self,
        uids,
        series_root,
        target_shape=(64, 448, 448),
        patch_size=112,
        patches_per_slice=1,
        regime_probs=None,
        p_identity=0.20,
        enable_motion=True,
        p_motion=0.15,
        hard_uid_set=None,
        hard_uid_repeat=1,
    ):
        self.uids = list(uids)
        self.series_root = series_root
        self.target_shape = target_shape
        self.patch_size = int(patch_size)
        self.patches_per_slice = int(patches_per_slice)
        self.regime_probs = _normalize_probs(regime_probs or REGIME_PROBS_STAGE_A)
        self.p_identity = float(p_identity)
        self.enable_motion = bool(enable_motion)
        self.p_motion = float(p_motion)
        self.cache = VolumeLRU(max_items=8)

        self.hard_uid_set = set(hard_uid_set or [])
        self.hard_uid_repeat = int(max(1, hard_uid_repeat))

        items = []
        for ui in range(len(self.uids)):
            uid = self.uids[ui]
            rep = self.hard_uid_repeat if uid in self.hard_uid_set else 1
            for _ in range(rep):
                items.extend([(ui, z) for z in range(1, target_shape[0] - 1)])
        self.items = items

    def __len__(self):
        return len(self.items)

    def _sample_patch_xy(self, cent, ps):
        H, W = cent.shape
        if random.random() < P_RANDOM_PATCH:
            return np.random.randint(0, H - ps + 1), np.random.randint(0, W - ps + 1)

        for _ in range(ANATOMY_REJECT_TRIES):
            y = np.random.randint(0, H - ps + 1)
            x = np.random.randint(0, W - ps + 1)
            patch = cent[y:y+ps, x:x+ps]
            if patch.mean() > ANATOMY_MEAN_TH and patch.std() > ANATOMY_STD_TH:
                return y, x

        return (H - ps) // 2, (W - ps) // 2

    def __getitem__(self, idx):
        ui, z = self.items[idx]
        uid = self.uids[ui]

        vol = self.cache.get(uid)
        if vol is None:
            vol = load_series_volume(uid, self.series_root, self.target_shape)
            if vol is None:
                return self.__getitem__(random.randint(0, len(self.items) - 1))
            self.cache.put(uid, vol)

        ps = self.patch_size
        y, x = self._sample_patch_xy(vol[z], ps)

        clean = vol[z][y:y+ps, x:x+ps].copy()
        pp = vol[z-1][y:y+ps, x:x+ps].copy()
        cc = vol[z][y:y+ps, x:x+ps].copy()
        nn_ = vol[z+1][y:y+ps, x:x+ps].copy()

        is_identity = 0.0
        if random.random() < self.p_identity:
            blur_level, dose_mode, do_motion, is_identity = 0, "clean", False, 1.0
        else:
            blur_level, dose_mode, do_motion = _sample_regime_params(self.regime_probs)
            do_motion = do_motion and self.enable_motion and (random.random() < self.p_motion)

        bp = gaussian_psf_surrogate(pp, blur_level, alpha=DIFFUSION_ALPHA)
        bc = gaussian_psf_surrogate(cc, blur_level, alpha=DIFFUSION_ALPHA)
        bn = gaussian_psf_surrogate(nn_, blur_level, alpha=DIFFUSION_ALPHA)

        if do_motion:
            L = random.choice([3, 5, 7, 9, 11])
            A = random.uniform(0, 180)
            bp = motion_artifact_surrogate(bp, L, A)
            bc = motion_artifact_surrogate(bc, L, A)
            bn = motion_artifact_surrogate(bn, L, A)

        if dose_mode != "clean":
            bp = mixed_poisson_gaussian(bp, dose_mode)
            bc = mixed_poisson_gaussian(bc, dose_mode)
            bn = mixed_poisson_gaussian(bn, dose_mode)

        cp = clean.copy()
        if random.random() > 0.5:
            cp, bp, bc, bn = cp[::-1].copy(), bp[::-1].copy(), bc[::-1].copy(), bn[::-1].copy()
        if random.random() > 0.5:
            cp, bp, bc, bn = cp[:, ::-1].copy(), bp[:, ::-1].copy(), bc[:, ::-1].copy(), bn[:, ::-1].copy()
        k = random.randint(0, 3)
        if k > 0:
            cp, bp, bc, bn = np.rot90(cp, k).copy(), np.rot90(bp, k).copy(), np.rot90(bc, k).copy(), np.rot90(bn, k).copy()

        t_norm = float(blur_level) / BLUR_LEVEL_MAX if BLUR_LEVEL_MAX > 0 else 0.0

        inp = np.stack([bp, bc, bn, np.full_like(bc, t_norm, dtype=np.float32)], axis=0).astype(np.float32)
        tgt = cp[np.newaxis, ...].astype(np.float32)

        meta = np.array([
            t_norm,
            1.0 if do_motion else 0.0,
            1.0 if dose_mode == "clean" else 0.0,
            is_identity
        ], dtype=np.float32)

        return (
            torch.from_numpy(inp).float(),
            torch.from_numpy(tgt).float(),
            torch.from_numpy(meta).float()
        )

class UIDBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, seed=42, batches_per_epoch=None):
        self.dataset = dataset
        self.batch_size = int(batch_size)
        self.rng = random.Random(seed)
        self.by_ui = {}

        for idx, (ui, z) in enumerate(dataset.items):
            self.by_ui.setdefault(ui, []).append(idx)

        self.ui_keys = list(self.by_ui.keys())
        self.batches_per_epoch = int(batches_per_epoch) if batches_per_epoch else len(dataset) // self.batch_size

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        for _ in range(self.batches_per_epoch):
            ui = self.rng.choice(self.ui_keys)
            pool = self.by_ui[ui]
            if len(pool) >= self.batch_size:
                yield self.rng.sample(pool, self.batch_size)
            else:
                yield [self.rng.choice(pool) for _ in range(self.batch_size)]

# ============================================================
# loss
# ============================================================

def charbonnier_loss(pred, target, eps=1e-3):
    return torch.mean(torch.sqrt((pred - target) ** 2 + eps ** 2))

def fft_spectrum_loss(pred, target, fft_mask):
    pred_fft = torch.fft.rfft2(pred.float(), dim=(-2, -1), norm="ortho")
    tgt_fft = torch.fft.rfft2(target.float(), dim=(-2, -1), norm="ortho")
    m = fft_mask.view(1, 1, *fft_mask.shape)
    return charbonnier_loss(torch.abs(pred_fft) * m, torch.abs(tgt_fft) * m)

def ssim_loss(pred, target, window_size=11):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    pad = window_size // 2

    mu_x = F.avg_pool2d(pred, window_size, stride=1, padding=pad)
    mu_y = F.avg_pool2d(target, window_size, stride=1, padding=pad)

    sigma_x2 = F.avg_pool2d(pred ** 2, window_size, stride=1, padding=pad) - mu_x ** 2
    sigma_y2 = F.avg_pool2d(target ** 2, window_size, stride=1, padding=pad) - mu_y ** 2
    sigma_xy = F.avg_pool2d(pred * target, window_size, stride=1, padding=pad) - mu_x * mu_y

    ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / (
        (mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x2 + sigma_y2 + C2) + 1e-8
    )
    return 1.0 - ssim_map.mean()

def _make_fft_mask(H, W, fcut=0.20, device="cpu"):
    fy = torch.fft.fftfreq(H, d=1.0, device=device).view(H, 1).abs()
    fx = torch.fft.rfftfreq(W, d=1.0, device=device).view(1, W // 2 + 1).abs()
    return (torch.sqrt(fx * fx + fy * fy) >= fcut).float()

class UltimatePhysicsLoss(nn.Module):
    def __init__(self, patch_size=112):
        super().__init__()
        self.sobel_x = torch.tensor([[[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]]).view(1, 1, 3, 3).to(device)
        self.sobel_y = torch.tensor([[[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]]).view(1, 1, 3, 3).to(device)
        self.lap = torch.tensor([[[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]]).view(1, 1, 3, 3).to(device)
        self.register_buffer("fft_mask", _make_fft_mask(patch_size, patch_size, fcut=FFT_FCUTOFF, device=device))

    def forward(self, pred, target, allow_fft=False):
        total = W_CHARBONNIER * charbonnier_loss(pred, target)
        total += W_SSIM * ssim_loss(pred, target)

        p_pad = F.pad(pred, (1, 1, 1, 1), mode="replicate")
        t_pad = F.pad(target, (1, 1, 1, 1), mode="replicate")

        total += W_SOBEL * (
            charbonnier_loss(F.conv2d(p_pad, self.sobel_x), F.conv2d(t_pad, self.sobel_x))
            + charbonnier_loss(F.conv2d(p_pad, self.sobel_y), F.conv2d(t_pad, self.sobel_y))
        )
        total += W_LAP * charbonnier_loss(F.conv2d(p_pad, self.lap), F.conv2d(t_pad, self.lap))

        if allow_fft:
            total += W_FFT * fft_spectrum_loss(pred, target, self.fft_mask)
        return total

def alg_humility_penalty(pred, inp, is_id, weight_change_id):
    center = inp[:, 1:2]
    per_sample = torch.mean(torch.abs(pred - center), dim=(1, 2, 3))
    return torch.mean(per_sample * is_id * weight_change_id)

def low_t_edit_penalty(pred, inp, meta, weight_low_t):
    center = inp[:, 1:2]
    t_norm = meta[:, 0]
    low_mask = (t_norm <= LOW_T_THR).float()
    per_sample = torch.mean(torch.abs(pred - center), dim=(1, 2, 3))
    return torch.mean(per_sample * low_mask * weight_low_t)

def authority_tv_penalty(authority, weight_auth_tv):
    dy = torch.abs(authority[:, :, 1:, :] - authority[:, :, :-1, :]).mean()
    dx = torch.abs(authority[:, :, :, 1:] - authority[:, :, :, :-1]).mean()
    return (dx + dy) * weight_auth_tv

# ============================================================
# validation
# ============================================================

@torch.no_grad()
def eval_model_psnr(model, val_uids, series_root, target_shape=(64, 448, 448), max_uids=8):
    model.eval()
    scores = []
    cache = VolumeLRU(max_items=2)

    for uid in list(val_uids)[:max_uids]:
        vol = cache.get(uid)
        if vol is None:
            vol = load_series_volume(uid, series_root, target_shape)
            if vol is None:
                continue
            cache.put(uid, vol)

        D = vol.shape[0]
        for blur_level in [3, 8]:
            for z in range(1, D - 1, 8):
                cl = vol[z].astype(np.float32)
                prev = vol[z - 1].astype(np.float32)
                cent = vol[z].astype(np.float32)
                next_ = vol[z + 1].astype(np.float32)

                bp = mixed_poisson_gaussian(gaussian_psf_surrogate(prev, blur_level, DIFFUSION_ALPHA), "quarter")
                bc = mixed_poisson_gaussian(gaussian_psf_surrogate(cent, blur_level, DIFFUSION_ALPHA), "quarter")
                bn = mixed_poisson_gaussian(gaussian_psf_surrogate(next_, blur_level, DIFFUSION_ALPHA), "quarter")

                t_norm = float(blur_level) / BLUR_LEVEL_MAX
                inp_np = np.stack([bp, bc, bn, np.full_like(bc, t_norm)], axis=0).astype(np.float32)
                meta_np = np.array([[t_norm, 0.0, 0.0, 0.0]], dtype=np.float32)

                inp_t = torch.from_numpy(inp_np).unsqueeze(0).to(device)
                meta_t = torch.from_numpy(meta_np).to(device)

                with AMP_CTX():
                    pred = model(inp_t, meta=meta_t)[0, 0].float().cpu().numpy()

                mse = float(np.mean((pred - cl) ** 2))
                scores.append(99.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse))

    return float(np.mean(scores)) if scores else None

# ============================================================
# hard-case mining
# only on refine split
# ============================================================

@torch.no_grad()
def mine_hard_uids(model, refine_uids, series_root, target_shape=(64, 448, 448), max_uids=None):
    model.eval()
    cache = VolumeLRU(max_items=2)
    rows = []

    use_uids = list(refine_uids)
    if max_uids is not None:
        use_uids = use_uids[:max_uids]

    for uid in use_uids:
        vol = cache.get(uid)
        if vol is None:
            vol = load_series_volume(uid, series_root, target_shape)
            if vol is None:
                continue
            cache.put(uid, vol)

        D = vol.shape[0]
        psnr_scores = []
        overedit_scores = []

        for blur_level in MINE_BLUR_LEVELS:
            for z in range(1, D - 1, MINE_STRIDE_Z):
                cl = vol[z].astype(np.float32)
                prev = vol[z - 1].astype(np.float32)
                cent = vol[z].astype(np.float32)
                next_ = vol[z + 1].astype(np.float32)

                bp = mixed_poisson_gaussian(gaussian_psf_surrogate(prev, blur_level, DIFFUSION_ALPHA), "quarter")
                bc = mixed_poisson_gaussian(gaussian_psf_surrogate(cent, blur_level, DIFFUSION_ALPHA), "quarter")
                bn = mixed_poisson_gaussian(gaussian_psf_surrogate(next_, blur_level, DIFFUSION_ALPHA), "quarter")

                t_norm = float(blur_level) / BLUR_LEVEL_MAX
                inp_np = np.stack([bp, bc, bn, np.full_like(bc, t_norm)], axis=0).astype(np.float32)
                meta_np = np.array([[t_norm, 0.0, 0.0, 0.0]], dtype=np.float32)

                inp_t = torch.from_numpy(inp_np).unsqueeze(0).to(device)
                meta_t = torch.from_numpy(meta_np).to(device)

                with AMP_CTX():
                    pred, aux = model(inp_t, meta=meta_t, return_aux=True)
                    pred_np = pred[0, 0].float().cpu().numpy()

                mse = float(np.mean((pred_np - cl) ** 2))
                psnr = 99.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse)
                psnr_scores.append(psnr)

                overedit = float(np.mean(np.abs(pred_np - bc)))
                overedit_scores.append(overedit)

        if len(psnr_scores) == 0:
            continue

        psnr_mean = float(np.mean(psnr_scores))
        overedit_mean = float(np.mean(overedit_scores))

        rows.append({
            "uid": uid,
            "psnr_mean": psnr_mean,
            "overedit_mean": overedit_mean,
        })

    df_hard = pd.DataFrame(rows)
    if len(df_hard) == 0:
        return [], df_hard

    # 分数越大越难：低 PSNR + 高 overedit
    psnr_norm = (df_hard["psnr_mean"].max() - df_hard["psnr_mean"])
    if psnr_norm.max() > 0:
        psnr_norm = psnr_norm / (psnr_norm.max() + 1e-8)

    over_norm = df_hard["overedit_mean"]
    if over_norm.max() > 0:
        over_norm = over_norm / (over_norm.max() + 1e-8)

    df_hard["hard_score"] = HARD_SCORE_W_PSNR * psnr_norm + HARD_SCORE_W_OVEREDIT * over_norm
    df_hard = df_hard.sort_values("hard_score", ascending=False).reset_index(drop=True)

    hard_uids = df_hard["uid"].tolist()[:min(N_HARD_UIDS, len(df_hard))]
    return hard_uids, df_hard

# ============================================================
# generic training loop
# ============================================================

def run_training_stage(
    stage_name,
    model,
    train_loader,
    val_uids,
    save_best,
    save_last,
    epochs,
    lr,
    weight_low_t,
    weight_auth_tv,
    extra_metadata=None,
):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = UltimatePhysicsLoss(patch_size=PATCH_SIZE).to(device)

    best_psnr = -1.0
    hist = []

    print(f"\n=== {stage_name} ===")
    print("lr:", lr, "| epochs:", epochs)

    for ep in range(1, epochs + 1):
        model.train()
        losses = []
        t0 = time.time()

        for b, (inp, tgt, meta) in enumerate(train_loader, 1):
            inp = inp.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            meta = meta.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with AMP_CTX():
                pred, aux = model(inp, meta=meta, return_aux=True)

                t_scalar = meta[:, 0].mean().item() * BLUR_LEVEL_MAX
                is_motion = meta[:, 1]
                is_id = meta[:, 3]

                allow_fft = (t_scalar <= FFT_ONLY_IF_T_LE) and not bool((is_motion > 0.5).any().item())

                loss_main = crit(pred, tgt, allow_fft=allow_fft)
                loss_id = alg_humility_penalty(pred, inp, is_id, W_CHANGE_ID)
                loss_lowt = low_t_edit_penalty(pred, inp, meta, weight_low_t)
                loss_auth = authority_tv_penalty(aux["authority"], weight_auth_tv)

                loss = loss_main + loss_id + loss_lowt + loss_auth

            if USE_AMP:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()

            losses.append(float(loss.item()))

            if b == 1 or b % 50 == 0 or b == len(train_loader):
                print(
                    f"  [{stage_name} | Epoch {ep:02d} | Batch {b:03d}/{len(train_loader)}] "
                    f"Loss={np.mean(losses[-20:]):.4f} "
                    f"(main={float(loss_main.item()):.4f}, id={float(loss_id.item()):.4f}, "
                    f"lowt={float(loss_lowt.item()):.4f}, auth={float(loss_auth.item()):.6f})"
                )

        sched.step()
        epoch_loss = float(np.mean(losses)) if len(losses) else np.nan
        print(f"{stage_name} Epoch {ep:02d} done | Loss={epoch_loss:.5f} | Time={(time.time()-t0)/60:.1f} min")

        row = {"stage": stage_name, "epoch": ep, "train_loss": epoch_loss}

        if ep % 3 == 0 or ep == epochs:
            psnr_val = eval_model_psnr(model, val_uids, RSNA_DATA_ROOT, (TARGET_D, TARGET_H, TARGET_W), max_uids=8)
            row["val_psnr"] = psnr_val

            if psnr_val is not None:
                if psnr_val > best_psnr:
                    best_psnr = psnr_val
                    pack = {
                        "model": model.state_dict(),
                        "epoch": ep,
                        "best_val_psnr": best_psnr,
                        "stage": stage_name,
                        "extra_metadata": extra_metadata or {},
                    }
                    torch.save(pack, save_best)
                    print(f"  [Val] PSNR={psnr_val:.2f} dB ★ NEW BEST")
                else:
                    print(f"  [Val] PSNR={psnr_val:.2f} dB")

        hist.append(row)

    final_pack = {
        "model": model.state_dict(),
        "epoch": epochs,
        "best_val_psnr": best_psnr,
        "stage": stage_name,
        "extra_metadata": extra_metadata or {},
    }
    torch.save(final_pack, save_last)

    return best_psnr, pd.DataFrame(hist)

# ============================================================
# 1) split data
# ============================================================

print("\n=== Step 1. Build scientific train/refine/val split ===")
train_uids, refine_uids, val_uids = build_ct_uid_lists_3way(
    META_CSV,
    RSNA_DATA_ROOT,
    TRAIN_LOCALIZERS_CSV,
    N_TRAIN_UIDS,
    N_REFINE_UIDS,
    N_VAL_UIDS,
    SEED,
)

pd.DataFrame({"SeriesInstanceUID": train_uids}).to_csv(OUT_TRAIN_UIDS, index=False)
pd.DataFrame({"SeriesInstanceUID": refine_uids}).to_csv(OUT_REFINE_UIDS, index=False)
pd.DataFrame({"SeriesInstanceUID": val_uids}).to_csv(OUT_VAL_UIDS, index=False)

print(f"train={len(train_uids)} | refine={len(refine_uids)} | val={len(val_uids)}")

# ============================================================
# 2) Stage A dataset
# ============================================================

print("\n=== Step 2. Stage A dataset ===")
train_ds_A = CTDeblur25D(
    train_uids,
    RSNA_DATA_ROOT,
    (TARGET_D, TARGET_H, TARGET_W),
    PATCH_SIZE,
    PATCHES_PER_SLICE,
    regime_probs=REGIME_PROBS_STAGE_A,
    p_identity=P_IDENTITY,
    enable_motion=ENABLE_MOTION,
    p_motion=P_MOTION,
    hard_uid_set=None,
    hard_uid_repeat=1,
)

train_loader_A = DataLoader(
    train_ds_A,
    batch_sampler=UIDBatchSampler(train_ds_A, BATCH_SIZE, SEED, BATCHES_PER_EPOCH_A),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# ============================================================
# 3) init model
# ============================================================

print("\n=== Step 3. Initialize model ===")
model = DeblurUNet25D_Ultimate(
    in_ch=4,
    out_ch=1,
    base=32,
    res_min=RES_MIN,
    res_max=RES_MAX,
    meta_dim=4,
    film_hidden=64,
    authority_bias_init=2.0,
).to(device)

print("Trainable params:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ============================================================
# 4) Stage A: base training
# ============================================================

stageA_meta = {
    "seed": SEED,
    "train_uids": train_uids,
    "refine_uids": refine_uids,
    "val_uids": val_uids,
    "regime_probs": REGIME_PROBS_STAGE_A,
    "phase": "base_training",
}

bestA, histA = run_training_stage(
    stage_name="StageA_Base",
    model=model,
    train_loader=train_loader_A,
    val_uids=val_uids,
    save_best=SAVE_STAGEA_BEST,
    save_last=SAVE_STAGEA_LAST,
    epochs=EPOCHS_STAGE_A,
    lr=LR_STAGE_A,
    weight_low_t=W_LOW_T_EDIT,
    weight_auth_tv=W_AUTH_TV,
    extra_metadata=stageA_meta,
)

histA.to_csv(os.path.join(OUTDIR, "history_stageA.csv"), index=False)

print("\nStage A best val PSNR:", bestA)

# ============================================================
# 5) Failure analysis on REFINE split only
# ============================================================

print("\n=== Step 5. Mine hard cases on refine split only ===")
if os.path.exists(SAVE_STAGEA_BEST):
    packA = torch.load(SAVE_STAGEA_BEST, map_location="cpu")
    model.load_state_dict(packA["model"], strict=True)
    model = model.to(device).eval()

hard_uids, df_hard = mine_hard_uids(
    model,
    refine_uids,
    RSNA_DATA_ROOT,
    (TARGET_D, TARGET_H, TARGET_W),
    max_uids=MINE_EVAL_MAX_UIDS,
)

df_hard.to_csv(os.path.join(OUTDIR, "refine_uid_difficulty.csv"), index=False)
pd.DataFrame({"SeriesInstanceUID": hard_uids}).to_csv(OUT_HARD_UIDS, index=False)

print(f"Hard UIDs selected for Stage B: {len(hard_uids)}")
display(df_hard.head(20))

# ============================================================
# 6) Stage B dataset
# only uses:
#   - original TRAIN split
#   - failure-derived hard UID list from REFINE split analysis
# no external test leakage
# ============================================================

print("\n=== Step 6. Stage B refinement dataset ===")
stageB_uids = list(train_uids) + list(hard_uids)

train_ds_B = CTDeblur25D(
    stageB_uids,
    RSNA_DATA_ROOT,
    (TARGET_D, TARGET_H, TARGET_W),
    PATCH_SIZE,
    PATCHES_PER_SLICE,
    regime_probs=REGIME_PROBS_STAGE_B,
    p_identity=P_IDENTITY,
    enable_motion=ENABLE_MOTION,
    p_motion=P_MOTION,
    hard_uid_set=set(hard_uids),
    hard_uid_repeat=HARD_UID_REPEAT,
)

train_loader_B = DataLoader(
    train_ds_B,
    batch_sampler=UIDBatchSampler(train_ds_B, BATCH_SIZE, SEED + 101, BATCHES_PER_EPOCH_B),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Stage B total UID list = {len(stageB_uids)} (train + hard refine)")
print(f"Hard UID repeat = {HARD_UID_REPEAT}")

# ============================================================
# 7) load Stage A best and do Stage B refinement
# ============================================================

print("\n=== Step 7. Stage B refinement ===")
if os.path.exists(SAVE_STAGEA_BEST):
    packA = torch.load(SAVE_STAGEA_BEST, map_location="cpu")
    model.load_state_dict(packA["model"], strict=True)
    model = model.to(device)

stageB_meta = {
    "seed": SEED,
    "train_uids": train_uids,
    "refine_uids": refine_uids,
    "val_uids": val_uids,
    "hard_uids": hard_uids,
    "regime_probs": REGIME_PROBS_STAGE_B,
    "phase": "failure_driven_refinement",
    "note": "hard cases mined only from refine split, not from external test set",
}

bestB, histB = run_training_stage(
    stage_name="StageB_Refine",
    model=model,
    train_loader=train_loader_B,
    val_uids=val_uids,
    save_best=SAVE_STAGEB_BEST,
    save_last=SAVE_STAGEB_LAST,
    epochs=EPOCHS_STAGE_B,
    lr=LR_STAGE_B,
    weight_low_t=W_LOW_T_EDIT_STAGE_B,
    weight_auth_tv=W_AUTH_TV_STAGE_B,
    extra_metadata=stageB_meta,
)

histB.to_csv(os.path.join(OUTDIR, "history_stageB.csv"), index=False)

# ============================================================
# 8) final summary
# ============================================================

summary = {
    "train_uids": len(train_uids),
    "refine_uids": len(refine_uids),
    "val_uids": len(val_uids),
    "hard_uids_stageB": len(hard_uids),
    "best_stageA_val_psnr": bestA,
    "best_stageB_val_psnr": bestB,
    "stageA_best_ckpt": SAVE_STAGEA_BEST,
    "stageB_best_ckpt": SAVE_STAGEB_BEST,
}

with open(os.path.join(OUTDIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 90)
print("Scientific-method training finished")
print("=" * 90)
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSaved files:")
for p in sorted(os.listdir(OUTDIR)):
    print(" -", os.path.join(OUTDIR, p))

print("\n✅ Weight File for later evaluation：")
print("   ", SAVE_STAGEB_BEST if os.path.exists(SAVE_STAGEB_BEST) else SAVE_STAGEA_BEST)

Device: cuda

=== Step 1. Build scientific train/refine/val split ===
train=300 | refine=60 | val=20

=== Step 2. Stage A dataset ===

=== Step 3. Initialize model ===
Trainable params: 2,326,186

=== StageA_Base ===
lr: 0.0002 | epochs: 18
  [StageA_Base | Epoch 01 | Batch 001/300] Loss=0.0590 (main=0.0585, id=0.0000, lowt=0.0005, auth=0.000000)
  [StageA_Base | Epoch 01 | Batch 050/300] Loss=0.0420 (main=0.0488, id=0.0000, lowt=0.0010, auth=0.000000)
  [StageA_Base | Epoch 01 | Batch 100/300] Loss=0.0402 (main=0.0415, id=0.0000, lowt=0.0001, auth=0.000000)
  [StageA_Base | Epoch 01 | Batch 150/300] Loss=0.0394 (main=0.0320, id=0.0000, lowt=0.0007, auth=0.000000)
  [StageA_Base | Epoch 01 | Batch 200/300] Loss=0.0421 (main=0.0337, id=0.0000, lowt=0.0002, auth=0.000000)
  [StageA_Base | Epoch 01 | Batch 250/300] Loss=0.0410 (main=0.0353, id=0.0000, lowt=0.0006, auth=0.000000)
  [StageA_Base | Epoch 01 | Batch 300/300] Loss=0.0410 (main=0.0368, id=0.0000, lowt=0.0009, auth=0.000000)
Sta

/usr/local/lib/python3.12/dist-packages/pydicom/pixels/utils.py:222: UserWarning: A value of 'None' for (0028,0008) 'Number of Frames' is invalid, assuming 1 frame
  warn_and_log(


  [StageA_Base | Epoch 03 | Batch 250/300] Loss=0.0402 (main=0.0473, id=0.0000, lowt=0.0009, auth=0.000001)
  [StageA_Base | Epoch 03 | Batch 300/300] Loss=0.0378 (main=0.0430, id=0.0000, lowt=0.0007, auth=0.000002)
StageA_Base Epoch 03 done | Loss=0.03913 | Time=4.4 min
  [Val] PSNR=38.36 dB ★ NEW BEST
  [StageA_Base | Epoch 04 | Batch 001/300] Loss=0.0399 (main=0.0388, id=0.0000, lowt=0.0010, auth=0.000002)


/usr/local/lib/python3.12/dist-packages/pydicom/pixels/utils.py:222: UserWarning: A value of 'None' for (0028,0008) 'Number of Frames' is invalid, assuming 1 frame
  warn_and_log(


  [StageA_Base | Epoch 04 | Batch 050/300] Loss=0.0361 (main=0.0312, id=0.0000, lowt=0.0007, auth=0.000002)
  [StageA_Base | Epoch 04 | Batch 100/300] Loss=0.0310 (main=0.0275, id=0.0000, lowt=0.0015, auth=0.000002)
  [StageA_Base | Epoch 04 | Batch 150/300] Loss=0.0313 (main=0.0254, id=0.0000, lowt=0.0009, auth=0.000002)
  [StageA_Base | Epoch 04 | Batch 200/300] Loss=0.0272 (main=0.0192, id=0.0000, lowt=0.0015, auth=0.000001)
  [StageA_Base | Epoch 04 | Batch 250/300] Loss=0.0281 (main=0.0297, id=0.0000, lowt=0.0013, auth=0.000001)
  [StageA_Base | Epoch 04 | Batch 300/300] Loss=0.0279 (main=0.0148, id=0.0000, lowt=0.0007, auth=0.000001)
StageA_Base Epoch 04 done | Loss=0.03057 | Time=4.2 min
  [StageA_Base | Epoch 05 | Batch 001/300] Loss=0.0181 (main=0.0160, id=0.0000, lowt=0.0022, auth=0.000001)
  [StageA_Base | Epoch 05 | Batch 050/300] Loss=0.0248 (main=0.0229, id=0.0000, lowt=0.0015, auth=0.000001)


/usr/local/lib/python3.12/dist-packages/pydicom/pixels/utils.py:222: UserWarning: A value of 'None' for (0028,0008) 'Number of Frames' is invalid, assuming 1 frame
  warn_and_log(


  [StageA_Base | Epoch 05 | Batch 100/300] Loss=0.0226 (main=0.0237, id=0.0000, lowt=0.0009, auth=0.000001)
  [StageA_Base | Epoch 05 | Batch 150/300] Loss=0.0233 (main=0.0205, id=0.0000, lowt=0.0016, auth=0.000001)
  [StageA_Base | Epoch 05 | Batch 200/300] Loss=0.0224 (main=0.0310, id=0.0000, lowt=0.0013, auth=0.000001)
  [StageA_Base | Epoch 05 | Batch 250/300] Loss=0.0229 (main=0.0188, id=0.0000, lowt=0.0010, auth=0.000000)
  [StageA_Base | Epoch 05 | Batch 300/300] Loss=0.0259 (main=0.0277, id=0.0000, lowt=0.0033, auth=0.000001)
StageA_Base Epoch 05 done | Loss=0.02429 | Time=4.3 min
  [StageA_Base | Epoch 06 | Batch 001/300] Loss=0.0243 (main=0.0206, id=0.0000, lowt=0.0038, auth=0.000001)
  [StageA_Base | Epoch 06 | Batch 050/300] Loss=0.0232 (main=0.0199, id=0.0000, lowt=0.0023, auth=0.000001)
  [StageA_Base | Epoch 06 | Batch 100/300] Loss=0.0216 (main=0.0236, id=0.0000, lowt=0.0016, auth=0.000001)
  [StageA_Base | Epoch 06 | Batch 150/300] Loss=0.0245 (main=0.0194, id=0.0000, 

In [ ]:
import os
BEST_CKPT_FOR_EVAL = SAVE_STAGEB_BEST if os.path.exists(SAVE_STAGEB_BEST) else SAVE_STAGEA_BEST
print("BEST_CKPT_FOR_EVAL =", BEST_CKPT_FOR_EVAL)


# Downstream Tests

These downstream experiments are not included simply to show that the images look cleaner. Their purpose is to test the **methodology** from several complementary angles.

The central question of this project is:

> **When true paired data are unavailable, can physics-grounded synthetic supervision train a restoration model that is clinically useful, externally transferable, and structurally trustworthy?**

A single metric cannot answer that question.  
A model may score well on an internal validation set yet fail on real data, or improve pixel-level quality while making unsafe or anatomically meaningless edits.  
For that reason, the downstream section is organized into multiple evidence blocks, each addressing a different reviewer concern.

---

## 1. Clinical Rescue Matrix (CRM)

### What it measures
CRM evaluates whether the restoration model can **recover clinically relevant signal** that was suppressed by degradation on a strict held-out OOD CT/CTA set.

Instead of asking only whether the image becomes smoother or sharper, CRM asks:

> **Can the model restore information that matters for a downstream clinical proxy task?**

### Why it matters
This is the most direct test of **functional usefulness**.  
If the synthetic supervision is genuinely valuable, the model should learn more than denoising or smoothing; it should recover image content that helps preserve or recover clinically meaningful signal.

### Core questions it answers
- Does the model help on a clinically motivated downstream proxy?
- Does it outperform or complement traditional image filters?
- Does it introduce harmful changes?
- Is its behavior conservative or overly aggressive?

### Role in the overall argument
CRM is the first evidence block:

> **It shows that the method is useful.**

In other words, it addresses **effectiveness**.

---

## 2. Mayo External Paired Evaluation

### What it measures
The Mayo experiment evaluates the model on **real external paired low-dose / full-dose CT**.

Here the comparison is made directly between:
- real Quarter Dose CT,
- the restored output,
- and real Full Dose CT as reference.

The key question is not whether the model performs well on data generated by the same synthetic process used in training, but rather:

> **Does training on synthetic physics-based pairs transfer to real low-dose clinical data?**

### Why it matters
This is the strongest evidence for **sim-to-real transfer** in the notebook.  
The training supervision was not built from true paired low-dose / high-dose scans of the same patient. It was created through physics-grounded degradation synthesis.  
Therefore, the critical test is whether this synthetic supervision teaches the model something that remains valid on real low-dose CT.

### Core questions it answers
- Is the synthetic supervision externally credible?
- Does the model improve over the original real low-dose input?
- Does it do so conservatively, rather than by aggressively rewriting the image?

### Role in the overall argument
Mayo is the second major evidence block:

> **It shows that the method transfers to the real world.**

In other words, it addresses **transferability**.

---

## 3. TotalSegmentator Anatomy Analysis

### What it measures
The TotalSegmentator analysis is not included to win a segmentation benchmark.  
Its purpose is to answer a more basic question:

> **Where does the model make changes?**

Specifically, it examines whether the restoration edits are:
- concentrated in anatomically relevant regions,
- associated with structures and boundaries,
- or merely distributed as uniform smoothing across the whole image.

### Why it matters
A major risk in medical image restoration is that a model may improve certain metrics by making broad, indiscriminate changes that are not anatomically meaningful.  
Such behavior can look visually attractive while still being scientifically or clinically untrustworthy.

TotalSegmentator provides an **anatomy-anchored view** of the edits:
- if changes are nearly absent, the model may be very conservative;
- if changes are measurable and concentrated around structures, that suggests the edits are not random;
- if segmentation agreement improves relative to Full Dose CT, then the restoration is likely doing something structurally meaningful rather than only smoothing pixels.

### Core questions it answers
- Are the edits anatomically grounded?
- Is the model performing structured restoration instead of global smoothing?
- Does the restored output become more anatomically consistent with the reference?

### Role in the overall argument
TotalSegmentator is the third evidence block:

> **It shows that the model’s edits are structured, interpretable, and therefore more trustworthy.**

In other words, it addresses **interpretability**.

---

## 4. Why these tests belong together

These downstream tests are not redundant.  
They form a progressive evidence chain:

### CRM
shows that the model has **functional value**.

### Mayo
shows that this value **transfers to real external low-dose CT**.

### TotalSegmentator
shows that the edits are **anatomically structured rather than arbitrary**.

Together, they support the broader methodological claim:

> **Physics-grounded synthetic supervision is not merely a way to generate images that resemble low-quality CT. It is a way to train restoration models that are useful, externally transferable, and structurally interpretable even when true paired data are missing.**

---

## How to read this section

- If you care most about **clinical usefulness**, focus on **CRM**.
- If you care most about **sim-to-real transfer**, focus on **Mayo**.
- If you care most about **where the model edits and why those edits should be trusted**, focus on **TotalSegmentator**.

All three sections support the same central idea:

> **The contribution of this project is not only a restoration model, but a method for training medical restoration systems under paired-data scarcity, together with downstream tests that evaluate whether that method deserves to be trusted.**


In [ ]:
# ============================================================
# Imports 
# ============================================================
import os, sys, gc, math, time, random, hashlib, inspect, shutil, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

try:
    from IPython.display import display
except Exception:
    display = print

try:
    cv2.setNumThreads(0)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (device.type == "cuda")
AMP_CTX = (lambda: torch.amp.autocast("cuda")) if USE_AMP else (lambda: nullcontext())

print("device:", device)
print("USE_AMP:", USE_AMP)
# ============================================================
#  Config
# ============================================================
from pathlib import Path
import os

RSNA_DATA_ROOT = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/series"
META_CSV = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv"
TRAIN_LOCALIZERS_CSV = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train_localizers.csv"  # optional

# ------------------------------------------------------------
# Final model checkpoint: use the best Stage-B scientific model
# ------------------------------------------------------------
CKPT_PATH = "/kaggle/input/datasets/linguoyuemma/deblur-stageb-best-pt/deblur_stageB_best.pt"

# Optional: split files from the scientific training pipeline
TRAIN_UIDS_CSV  = "/kaggle/input/datasets/linguoyuemma/refine-uids/train_uids.csv"
REFINE_UIDS_CSV = "/kaggle/input/datasets/linguoyuemma/refine-uids/refine_uids.csv"
VAL_UIDS_CSV    = "/kaggle/input/datasets/linguoyuemma/refine-uids/val_uids.csv"

# Clinical Judge (9th place flayer)
PREDICTION_PY = "/kaggle/input/datasets/mingzeli2009/rsna-prediction/prediction.py"
MODEL_BASE = "/kaggle/input/models/tom99763/9th-place-models-rsna-iad/pytorch/default/1"
FLAYER_DIR = f"{MODEL_BASE}/flayer/outputs_heatmap_aux_v1_acc2"

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUTDIR = Path("/kaggle/working/vultimate_scientific_eval")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
TARGET_D, TARGET_H, TARGET_W = 64, 448, 448
TARGET_SHAPE = (TARGET_D, TARGET_H, TARGET_W)

HU_MIN, HU_MAX = -1024.0, 3072.0
HU_RANGE = HU_MAX - HU_MIN
KEEP_MODALITIES = {"CT", "CTA"}

# ------------------------------------------------------------
# Degradation settings for evaluation
# Keep these aligned with the training-time degradation model
# ------------------------------------------------------------
EVAL_T = 8.0
EVAL_DOSE = "quarter"
DIFFUSION_ALPHA = 0.20
LAM = DIFFUSION_ALPHA          # backward compatibility for older cells
BLUR_T_MAX = 8.0
RESTORE_BATCH = 16

# ------------------------------------------------------------
# Evaluation scale
# ------------------------------------------------------------
SEED_CASES = 2026

N_PILOT_COMPARE = 10      # quick sanity check
N_OOD_EVAL = 50           # main CRM size; can later increase to 100/200
N_MC_CASES = 100          # Monte Carlo stability test
MC_SEEDS = [10, 42, 23, 55, 83, 9999, 7, 11, 19, 29]

# ------------------------------------------------------------
# TotalSegmentator
# Set this to the actual number you want to run
# ------------------------------------------------------------
N_TOTALSEG_CASES = 40
TOTALSEG_TASK = "total"


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------
print("CKPT_PATH exists:", os.path.exists(CKPT_PATH))
print("TRAIN_UIDS_CSV exists:", os.path.exists(TRAIN_UIDS_CSV))
print("REFINE_UIDS_CSV exists:", os.path.exists(REFINE_UIDS_CSV))
print("VAL_UIDS_CSV exists:", os.path.exists(VAL_UIDS_CSV))
print("RSNA_DATA_ROOT exists:", os.path.exists(RSNA_DATA_ROOT))
print("META_CSV exists:", os.path.exists(META_CSV))
print("OUTDIR:", OUTDIR)
# ============================================================
# Load NEW model checkpoint + Clinical Judge
# ============================================================
assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"

ckpt = torch.load(CKPT_PATH, map_location="cpu")
state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

# --- NEW V-Ultimate (FiLM + authority map) ---
model_25d = DeblurUNet25D_Ultimate(
    in_ch=4,
    out_ch=1,
    base=32,
    res_min=0.02,
    res_max=0.15,
    meta_dim=4,
    film_hidden=64,
    authority_bias_init=2.0,
).to(device)

model_25d.load_state_dict(state_dict, strict=True)
model_25d.eval()

print("✅ NEW V-Ultimate model loaded.")
print("has auth_head?", hasattr(model_25d, "auth_head"))
print("has film_e1?", hasattr(model_25d, "film_e1"))
print("train_uids in ckpt:", len(ckpt.get("train_uids", [])) if isinstance(ckpt, dict) else "NA")
print("best_val_psnr:", ckpt.get("best_val_psnr", "NA") if isinstance(ckpt, dict) else "NA")

# ---- Load clinical judge ----
import importlib.util
assert os.path.exists(PREDICTION_PY), f"prediction.py not found: {PREDICTION_PY}"

spec = importlib.util.spec_from_file_location("prediction", PREDICTION_PY)
pred_mod = importlib.util.module_from_spec(spec)
sys.modules["prediction"] = pred_mod
spec.loader.exec_module(pred_mod)

classifier = pred_mod.FlayerClassifier(flayer_dir=FLAYER_DIR)
classifier.load()

@torch.no_grad()
def aneurysm_predict(volume_uint8):
    return float(classifier.predict(volume_uint8)["aneurysm_prob"])

print("✅ Clinical Judge loaded.")

# ============================================================
# Minimal bridge + CRM utilities
# Run this BEFORE the CRM cell
# ============================================================

import os
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Output dirs
# ------------------------------------------------------------
CRM_OUTDIR = OUTDIR / "clinical_rescue_matrix"
CRM_OUTDIR.mkdir(parents=True, exist_ok=True)

MAYO_OUTDIR = OUTDIR / "mayo_external_eval"
MAYO_OUTDIR.mkdir(parents=True, exist_ok=True)

TOTALSEG_OUTDIR = OUTDIR / "totalseg"
TOTALSEG_OUTDIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) Load UID splits -> dev_uid_set
# ------------------------------------------------------------
def read_uid_csv(path):
    if path is None or not os.path.exists(path):
        return []
    df = pd.read_csv(path)
    if df.shape[1] == 0:
        return []
    return df.iloc[:, 0].astype(str).tolist()

train_uids = read_uid_csv(TRAIN_UIDS_CSV)
refine_uids = read_uid_csv(REFINE_UIDS_CSV)
val_uids = read_uid_csv(VAL_UIDS_CSV)

train_uid_set = set(train_uids)
refine_uid_set = set(refine_uids)
val_uid_set = set(val_uids)
dev_uid_set = train_uid_set | refine_uid_set | val_uid_set

print("train_uid_set :", len(train_uid_set))
print("refine_uid_set:", len(refine_uid_set))
print("val_uid_set   :", len(val_uid_set))
print("dev_uid_set   :", len(dev_uid_set))

# ------------------------------------------------------------
# 3) Small helpers
# ------------------------------------------------------------
def stable_uid_seed(uid: str) -> int:
    s = str(uid).encode("utf-8")
    return int(hashlib.md5(s).hexdigest()[:8], 16)

def uid_tail4(uid: str) -> str:
    return str(uid)[-4:]

def _clip01(x):
    return np.clip(np.asarray(x, dtype=np.float32), 0.0, 1.0).astype(np.float32)

def vol01_to_flayer_uint8(vol01):
    vol01 = _clip01(vol01)
    return (vol01 * 255.0).astype(np.uint8)

# ------------------------------------------------------------
# 4) CRM metrics
# ------------------------------------------------------------
def calc_abs_gain(p_gt, p_deg, p_rec):
    """
    Raw signed change in model score after restoration.
    Positive means p_rec > p_deg, negative means p_rec < p_deg.
    """
    return float(p_rec - p_deg)

def calc_target_gain(p_gt, p_deg, p_rec):
    """
    Improvement toward the ground-truth score.
    Positive means restoration moved closer to p_gt.
    """
    return float(abs(p_deg - p_gt) - abs(p_rec - p_gt))

def is_iatrogenic(p_gt, p_deg, p_rec, eps=1e-8):
    """
    Restoration is harmful if it moves farther away from p_gt than the degraded input.
    """
    return bool((abs(p_rec - p_gt) - abs(p_deg - p_gt)) > eps)

print("✅ CRM bridge utilities ready.")
print("CRM_OUTDIR:", CRM_OUTDIR)

In [ ]:
# ============================================================
# Clinical Rescue Matrix — streamlined evaluation-only version
# STRICT held-out RSNA CT/CTA OOD evaluation
# Depends on:
#   - model_25d
#   - RSNA_DATA_ROOT, META_CSV, CRM_OUTDIR
#   - dev_uid_set (from bridge cell)
#   - aneurysm_predict, load_series_volume, stable_uid_seed, uid_tail4
#   - _clip01, calc_abs_gain, calc_target_gain, is_iatrogenic
#   - vol01_to_flayer_uint8
# ============================================================

import os, gc, time, math, random
from pathlib import Path
from contextlib import nullcontext

import cv2
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except Exception:
    display = print

try:
    cv2.setNumThreads(0)
except Exception:
    pass

# ------------------------------------------------------------
# 0) Checks / config
# ------------------------------------------------------------
required_globals = [
    "model_25d", "device", "AMP_CTX",
    "RSNA_DATA_ROOT", "META_CSV", "CRM_OUTDIR",
    "dev_uid_set",
    "load_series_volume", "stable_uid_seed", "uid_tail4",
    "_clip01", "calc_abs_gain", "calc_target_gain", "is_iatrogenic",
    "vol01_to_flayer_uint8", "aneurysm_predict",
]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals/functions: {missing}")

OUTDIR = Path(CRM_OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET_D = int(globals().get("TARGET_D", 64))
TARGET_H = int(globals().get("TARGET_H", 448))
TARGET_W = int(globals().get("TARGET_W", 448))
TARGET_SHAPE = (TARGET_D, TARGET_H, TARGET_W)

HU_MIN = float(globals().get("HU_MIN", -1024.0))
HU_MAX = float(globals().get("HU_MAX", 3072.0))
HU_RANGE = HU_MAX - HU_MIN

DIFFUSION_ALPHA = float(globals().get("DIFFUSION_ALPHA", globals().get("LAM", 0.20)))
BLUR_T_MAX = float(globals().get("BLUR_T_MAX", 8.0))
RESTORE_BATCH = int(globals().get("RESTORE_BATCH", 16))

EVAL_T = float(globals().get("EVAL_T", 8.0))
EVAL_DOSE = str(globals().get("EVAL_DOSE", "quarter"))
N_CRM_CASES = int(globals().get("N_OOD_EVAL", 50))
SEED_CRM = 2026

GAIN_POS_TH = 0.005
GAIN_NEG_TH = -0.005

GAUSS_SIGMA = 0.8
MEDIAN_KSIZE = 3
BILATERAL_D = 7
BILATERAL_SIGMACOLOR = 35
BILATERAL_SIGMASPACE = 35
UNSHARP_SIGMA = 1.0
UNSHARP_AMOUNT = 0.8
USE_NLM = False
NLM_H = 7

RAW_CSV = OUTDIR / f"crm_raw_N{N_CRM_CASES}.csv"
SUM_CSV = OUTDIR / f"crm_summary_N{N_CRM_CASES}.csv"
PAIR_CSV = OUTDIR / f"crm_paired_vs_vultimate_N{N_CRM_CASES}.csv"
BUCKET_CSV = OUTDIR / f"crm_bucket_breakdown_N{N_CRM_CASES}.csv"
FIXED_UID_CSV = OUTDIR / f"crm_eval_uids_fixed_N{N_CRM_CASES}.csv"
USED_UID_CSV = OUTDIR / f"crm_used_eval_uids_N{N_CRM_CASES}.csv"

METHODS = [
    ("Degraded", "degraded"),
    ("Gaussian", "gaussian"),
    ("Median3", "median3"),
    ("Bilateral", "bilateral"),
    ("Unsharp", "unsharp"),
]
if USE_NLM:
    METHODS.append(("NLM", "nlm"))
METHODS.append(("V-Ultimate", "vultimate"))

print("Methods:", [m[0] for m in METHODS])

# ------------------------------------------------------------
# 1) Helpers
# ------------------------------------------------------------
def psnr01_full(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    mse = float(np.mean((a - b) ** 2))
    return 99.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse)

def crm_outcome_label(p_gt, p_deg, p_rec, abs_gain):
    if is_iatrogenic(p_gt, p_deg, p_rec):
        return "⚠️ Iatrogenic"
    if abs_gain > GAIN_POS_TH:
        return "✅ Successful Rescue"
    if abs_gain < GAIN_NEG_TH:
        return "⏬ Minor Deviation"
    return "➖ Algorithmic Humility"

def crm_bucket(p_gt, p_deg, tau=0.03):
    if p_gt >= 0.5 and p_deg < p_gt - tau:
        return "harmed_positive"
    if p_gt >= 0.5 and p_deg > p_gt + tau:
        return "overcall_positive"
    return "neutral_other"

def hu01_to_hu(vol01):
    return np.asarray(vol01, dtype=np.float32) * HU_RANGE + HU_MIN

def hu_to_01(hu):
    return np.clip((hu - HU_MIN) / HU_RANGE, 0.0, 1.0).astype(np.float32)

def window_hu_to_uint8(hu, center=40.0, width=400.0):
    x = np.clip((hu - (center - width / 2.0)) / (width + 1e-6), 0.0, 1.0)
    return (x * 255.0).astype(np.uint8)

def degrade_volume_fixed_uid(vol01, t, uid, dose_mode="quarter"):
    local_seed = stable_uid_seed(uid)
    py_state, np_state = random.getstate(), np.random.get_state()
    random.seed(local_seed)
    np.random.seed(local_seed % (2**32 - 1))

    sigma = math.sqrt(max(1e-8, 2.0 * DIFFUSION_ALPHA * float(t)))
    out = np.empty_like(vol01, dtype=np.float32)

    for z in range(vol01.shape[0]):
        x = cv2.GaussianBlur(
            vol01[z].astype(np.float32),
            (0, 0),
            sigmaX=sigma,
            sigmaY=sigma,
            borderType=cv2.BORDER_REPLICATE,
        )
        if dose_mode != "clean":
            peak = random.uniform(3000.0, 6000.0)
            sigma_e = random.uniform(0.01, 0.02)
            x = np.random.poisson(np.clip(x * peak, 0, None)).astype(np.float32) / peak
            x = x + np.random.randn(*x.shape).astype(np.float32) * sigma_e
        out[z] = np.clip(x, 0.0, 1.0).astype(np.float32)

    random.setstate(py_state)
    np.random.set_state(np_state)
    return out

def trad_identity(vol):
    return np.asarray(vol, dtype=np.float32).copy()

def trad_gaussian(vol, sigma=GAUSS_SIGMA):
    out = np.empty_like(vol, dtype=np.float32)
    for z in range(vol.shape[0]):
        out[z] = _clip01(cv2.GaussianBlur(
            vol[z], (0, 0), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REPLICATE
        ))
    return out

def trad_median(vol, ksize=MEDIAN_KSIZE):
    out = np.empty_like(vol, dtype=np.float32)
    for z in range(vol.shape[0]):
        x8 = (np.clip(vol[z], 0, 1) * 255).astype(np.uint8)
        out[z] = cv2.medianBlur(x8, ksize).astype(np.float32) / 255.0
    return out

def trad_bilateral(vol, d=BILATERAL_D, sigmaColor=BILATERAL_SIGMACOLOR, sigmaSpace=BILATERAL_SIGMASPACE):
    out = np.empty_like(vol, dtype=np.float32)
    for z in range(vol.shape[0]):
        x8 = (np.clip(vol[z], 0, 1) * 255).astype(np.uint8)
        out[z] = cv2.bilateralFilter(
            x8, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace
        ).astype(np.float32) / 255.0
    return out

def trad_unsharp(vol, sigma=UNSHARP_SIGMA, amount=UNSHARP_AMOUNT):
    out = np.empty_like(vol, dtype=np.float32)
    for z in range(vol.shape[0]):
        x = vol[z].astype(np.float32)
        blur = cv2.GaussianBlur(x, (0, 0), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REPLICATE)
        out[z] = np.clip(x + amount * (x - blur), 0.0, 1.0)
    return out

def trad_nlm(vol, h=NLM_H, center=40.0, width=400.0):
    vol_hu = hu01_to_hu(vol)
    out = np.empty_like(vol, dtype=np.float32)
    for z in range(vol.shape[0]):
        u8 = window_hu_to_uint8(vol_hu[z], center=center, width=width)
        den = cv2.fastNlMeansDenoising(u8, None, h=h, templateWindowSize=7, searchWindowSize=21)
        den01 = den.astype(np.float32) / 255.0
        den_hu = den01 * width + (center - width / 2.0)
        out[z] = hu_to_01(den_hu)
    return out

TRAD_METHODS = {
    "degraded": trad_identity,
    "gaussian": trad_gaussian,
    "median3": trad_median,
    "bilateral": trad_bilateral,
    "unsharp": trad_unsharp,
    "nlm": trad_nlm,
}

@torch.no_grad()
def run_vultimate(vol_deg01, t=EVAL_T, restore_batch=RESTORE_BATCH):
    vol_deg01 = np.asarray(vol_deg01, dtype=np.float32)
    D = vol_deg01.shape[0]
    out = vol_deg01.copy()

    t_norm = np.float32(0.0 if t <= 0 else float(t) / float(BLUR_T_MAX))
    meta_row = np.array([t_norm, 0.0, 0.0, 0.0], dtype=np.float32)

    for s in range(0, D, restore_batch):
        zs = list(range(s, min(D, s + restore_batch)))
        inp_batch = []
        for z in zs:
            bp = vol_deg01[max(0, z - 1)]
            bc = vol_deg01[z]
            bn = vol_deg01[min(D - 1, z + 1)]
            inp_batch.append(np.stack([bp, bc, bn, np.full_like(bc, t_norm)], axis=0).astype(np.float32))

        inp_t = torch.from_numpy(np.stack(inp_batch, axis=0)).to(device, non_blocking=True)
        meta_t = torch.from_numpy(np.repeat(meta_row[None, :], len(zs), axis=0)).to(device, non_blocking=True)

        with AMP_CTX():
            pred_obj = model_25d(inp_t, meta=meta_t)
            pred = pred_obj[0] if isinstance(pred_obj, (tuple, list)) else pred_obj
            pred = pred.float().cpu().numpy()[:, 0]

        for k, z in enumerate(zs):
            out[z] = _clip01(pred[k])

    return out

def apply_method(method_key, deg):
    if method_key == "vultimate":
        return run_vultimate(deg, t=EVAL_T, restore_batch=RESTORE_BATCH)
    if method_key in TRAD_METHODS:
        return TRAD_METHODS[method_key](deg)
    raise ValueError(f"Unknown method_key: {method_key}")

# ------------------------------------------------------------
# 2) Build strict held-out OOD pool
# ------------------------------------------------------------
print("=== Clinical Rescue Matrix | STRICT RSNA OOD CT/CTA ===")

meta = pd.read_csv(META_CSV)
ct_uids = set(
    meta.loc[meta["Modality"].astype(str).isin({"CT", "CTA"}), "SeriesInstanceUID"].astype(str).tolist()
)

all_series_dirs = [
    u for u in os.listdir(RSNA_DATA_ROOT)
    if os.path.isdir(os.path.join(RSNA_DATA_ROOT, u))
]

ood_pool = [u for u in all_series_dirs if (u in ct_uids) and (u not in dev_uid_set)]

print("\n=== Exclusion summary ===")
print("dev_uid_set :", len(dev_uid_set))
print("ood_pool    :", len(ood_pool))
print("overlap     :", len(set(ood_pool) & dev_uid_set), "(should be 0)")

if FIXED_UID_CSV.exists():
    crm_eval_uids = pd.read_csv(FIXED_UID_CSV)["uid_full"].astype(str).tolist()
    crm_eval_uids = [u for u in crm_eval_uids if u in ood_pool]
    print(f"Loaded fixed eval UID list: {len(crm_eval_uids)}")
else:
    random.Random(SEED_CRM).shuffle(ood_pool)
    crm_eval_uids = ood_pool[:N_CRM_CASES]
    pd.DataFrame({"uid_full": crm_eval_uids}).to_csv(FIXED_UID_CSV, index=False)
    print(f"Saved fixed eval UID list: {len(crm_eval_uids)}")

# ------------------------------------------------------------
# 3) Main loop
# ------------------------------------------------------------
rows = []
used_eval_uids = []
t0_all = time.time()

for i, uid in enumerate(crm_eval_uids, 1):
    case_t0 = time.time()

    try:
        try:
            vol = load_series_volume(uid, RSNA_DATA_ROOT, TARGET_SHAPE)
        except TypeError:
            try:
                vol = load_series_volume(uid, RSNA_DATA_ROOT)
            except TypeError:
                vol = load_series_volume(uid)
    except Exception:
        vol = None

    if vol is None:
        continue

    gt = vol.astype(np.float32)
    deg = degrade_volume_fixed_uid(gt, EVAL_T, uid, dose_mode=EVAL_DOSE)

    p_gt = float(aneurysm_predict(vol01_to_flayer_uint8(gt)))
    p_deg = float(aneurysm_predict(vol01_to_flayer_uint8(deg)))
    bucket = crm_bucket(p_gt, p_deg, tau=0.03)
    uid4 = uid_tail4(uid)

    print(f"\n[{len(used_eval_uids)+1:03d}/{len(crm_eval_uids)}] UID:{uid4} | GT:{p_gt:.4f} -> Deg:{p_deg:.4f} | bucket={bucket}")

    case_ok = False
    for m_name, m_key in METHODS:
        try:
            rec = apply_method(m_key, deg)
            p_rec = float(aneurysm_predict(vol01_to_flayer_uint8(rec)))
            abs_gain = float(calc_abs_gain(p_gt, p_deg, p_rec))
            tgt_gain = float(calc_target_gain(p_gt, p_deg, p_rec))
            iatro = int(is_iatrogenic(p_gt, p_deg, p_rec))
            psnr_val = float(psnr01_full(rec, gt))
            outcome = crm_outcome_label(p_gt, p_deg, p_rec, abs_gain)

            rows.append({
                "uid_full": uid,
                "uid4": uid4,
                "bucket": bucket,
                "method": m_name,
                "method_key": m_key,
                "p_gt": p_gt,
                "p_deg": p_deg,
                "p_rec": p_rec,
                "abs_gain": abs_gain,
                "target_gain": tgt_gain,
                "iatrogenic": iatro,
                "outcome": outcome,
                "psnr_db": psnr_val,
                "eval_t": EVAL_T,
                "eval_dose": EVAL_DOSE,
            })

            tag = (
                "✅ rescue" if tgt_gain > GAIN_POS_TH
                else "⚠️ negative" if tgt_gain < GAIN_NEG_TH
                else "➖ identity"
            )
            print(
                f"  ├─ {m_name:<10s} | Rec:{p_rec:.4f} | "
                f"TGain:{tgt_gain:+.4f} | AGain:{abs_gain:+.4f} | {outcome} | {tag}"
            )
            case_ok = True

            if m_key != "degraded":
                del rec

        except Exception as e:
            rows.append({
                "uid_full": uid,
                "uid4": uid4,
                "bucket": bucket,
                "method": m_name,
                "method_key": m_key,
                "error": repr(e),
            })
            print(f"  ├─ {m_name:<10s} | ERROR: {repr(e)}")

    if case_ok:
        used_eval_uids.append(uid)

    print(f"  -> case done in {time.time() - case_t0:.1f}s")

    del gt, deg, vol
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame({"uid_full": used_eval_uids}).to_csv(USED_UID_CSV, index=False)

print(f"\nAll CRM cases done. elapsed={(time.time() - t0_all)/60:.1f} min")
print(f"Valid cases: {len(used_eval_uids)}")

# ------------------------------------------------------------
# 4) Save + summary
# ------------------------------------------------------------
df = pd.DataFrame(rows)
df.to_csv(RAW_CSV, index=False)
if len(df) == 0:
    raise RuntimeError("No CRM results generated.")

df_ok = df.dropna(subset=["target_gain", "abs_gain", "psnr_db"]).copy()

summary_rows = []
for m_name, _ in METHODS:
    sub = df_ok[df_ok["method"] == m_name].copy()
    if len(sub) == 0:
        continue

    oc = sub["outcome"].value_counts()
    summary_rows.append({
        "Method": m_name,
        "N": int(len(sub)),
        "Mean Target Gain": round(float(sub["target_gain"].mean()), 4),
        "Target Win %": round(float((sub["target_gain"] > GAIN_POS_TH).mean() * 100), 1),
        "Target Neg %": round(float((sub["target_gain"] < GAIN_NEG_TH).mean() * 100), 1),
        "Mean Abs Gain": round(float(sub["abs_gain"].mean()), 4),
        "Iatrogenic %": round(float(sub["iatrogenic"].mean() * 100), 1),
        "PSNR (dB)": round(float(sub["psnr_db"].mean()), 2),
        "CRM_Rescue": f"{int(oc.get('✅ Successful Rescue', 0))} ({100.0 * oc.get('✅ Successful Rescue', 0) / len(sub):.1f}%)",
        "CRM_Humble": f"{int(oc.get('➖ Algorithmic Humility', 0))} ({100.0 * oc.get('➖ Algorithmic Humility', 0) / len(sub):.1f}%)",
        "CRM_Deviate": f"{int(oc.get('⏬ Minor Deviation', 0))} ({100.0 * oc.get('⏬ Minor Deviation', 0) / len(sub):.1f}%)",
        "CRM_Iatro": f"{int(oc.get('⚠️ Iatrogenic', 0))} ({100.0 * oc.get('⚠️ Iatrogenic', 0) / len(sub):.1f}%)",
    })

df_sum = pd.DataFrame(summary_rows)
df_sum.to_csv(SUM_CSV, index=False)

print("\n=== Clinical Rescue Matrix — Summary ===")
display(df_sum)

# bucket breakdown
bucket_rows = []
for bucket in sorted(df_ok["bucket"].dropna().unique()):
    for m_name, _ in METHODS:
        sub = df_ok[(df_ok["bucket"] == bucket) & (df_ok["method"] == m_name)].copy()
        if len(sub) == 0:
            continue
        bucket_rows.append({
            "bucket": bucket,
            "method": m_name,
            "N": int(len(sub)),
            "mean_target_gain": round(float(sub["target_gain"].mean()), 4),
            "mean_abs_gain": round(float(sub["abs_gain"].mean()), 4),
            "iatrogenic_%": round(float(sub["iatrogenic"].mean() * 100), 1),
            "psnr_db": round(float(sub["psnr_db"].mean()), 2),
        })

df_bucket = pd.DataFrame(bucket_rows)
df_bucket.to_csv(BUCKET_CSV, index=False)

print("\n=== Bucket breakdown ===")
display(df_bucket)

# paired vs V-Ultimate
if "V-Ultimate" in set(df_ok["method"].unique()):
    base = df_ok[df_ok["method"] == "V-Ultimate"][[
        "uid_full", "target_gain", "abs_gain", "iatrogenic", "psnr_db"
    ]].rename(columns={
        "target_gain": "tgain_base",
        "abs_gain": "again_base",
        "iatrogenic": "iatro_base",
        "psnr_db": "psnr_base",
    })

    paired_rows = []
    for m_name, _ in METHODS:
        if m_name == "V-Ultimate":
            continue

        cmp_df = df_ok[df_ok["method"] == m_name][[
            "uid_full", "target_gain", "abs_gain", "iatrogenic", "psnr_db"
        ]].rename(columns={
            "target_gain": "tgain_cmp",
            "abs_gain": "again_cmp",
            "iatrogenic": "iatro_cmp",
            "psnr_db": "psnr_cmp",
        })

        merged = base.merge(cmp_df, on="uid_full", how="inner")
        if len(merged) == 0:
            continue

        dt = merged["tgain_base"] - merged["tgain_cmp"]
        paired_rows.append({
            "vs": m_name,
            "N": int(len(merged)),
            "V wins TGain %": round(float((dt > 0.001).mean() * 100), 1),
            "V loses TGain %": round(float((dt < -0.001).mean() * 100), 1),
            "ΔTGain": round(float(dt.mean()), 4),
            "ΔPSNR": round(float((merged["psnr_base"] - merged["psnr_cmp"]).mean()), 3),
        })

    df_pair = pd.DataFrame(paired_rows)
    df_pair.to_csv(PAIR_CSV, index=False)

    print("\n=== Paired comparison (baseline: V-Ultimate) ===")
    display(df_pair)

print("\nSaved:")
print(" raw      ->", RAW_CSV)
print(" summary  ->", SUM_CSV)
print(" bucket   ->", BUCKET_CSV)
print(" pair     ->", PAIR_CSV if PAIR_CSV.exists() else "(not generated)")
print(" fixedUID ->", FIXED_UID_CSV)
print(" usedUID  ->", USED_UID_CSV)

## Results Interpretation — Clinical Rescue Matrix (Strict RSNA OOD)

### 1. What question does this experiment actually answer?

The Clinical Rescue Matrix (CRM) is not designed to test which method produces the smoothest image. Instead, it asks:

> When degradation suppresses clinically relevant signal, can a restoration method **recover the lost target signal** while minimizing harmful deviations (iatrogenic change)?

For that reason, the key metrics here are not PSNR alone, but:

- **Mean Target Gain**: average clinically relevant recovery
- **Target Win %**: fraction of cases with clear positive rescue
- **Iatrogenic %**: fraction of cases with harmful deviation
- **Bucket breakdown**: which kinds of cases each method actually helps

---

### 2. Overall result: V-Ultimate is the strongest overall method, but not a universal winner on every single case

Across 50 strict held-out OOD CT/CTA cases, V-Ultimate achieved:

- **Mean Target Gain = 0.0324** (**highest** among all methods)
- **Target Win % = 66.0%** (**highest**)
- **Iatrogenic % = 32.0%** (**tied lowest** with Bilateral)
- **PSNR = 42.74 dB** (**substantially higher** than all traditional baselines)
- **Successful Rescue = 31/50 = 62.0%** (**highest**)

This means that a model trained only with synthetic supervision is already capable of producing the **strongest overall clinical proxy benefit** on strict held-out OOD data, while maintaining a relatively low harmful-modification rate.

The most important point is not that V-Ultimate wins every case, but that:

> **At the population level, V-Ultimate gives the best combination of benefit, win rate, and safety.**

---

### 3. Gaussian is not weak, but it is not the end of the story

Gaussian is a strong baseline in this benchmark:

- Mean Target Gain = **0.0125**
- Target Win % = **50.0%**
- Iatrogenic % = **44.0%**

This shows that simple smoothing can help under some degradation patterns. Therefore, these results should **not** be interpreted as:

> “Traditional methods are useless, so deep learning is always necessary.”

A more accurate interpretation is:

> **Simple smoothing can recover signal in some cases, but V-Ultimate achieves higher overall benefit, higher win rate, and better safety, suggesting that it learns something more targeted than generic denoising.**

In other words, Gaussian serves as a strong control condition. It helps distinguish whether the model is doing:

- ordinary smoothing, or
- more targeted, controlled restoration

---

### 4. The most important finding: V-Ultimate is strongest where degradation truly harms the target

The most important CRM bucket is **`harmed_positive`**, meaning:

> the original high-quality scan contains a strong target signal, but degradation clearly suppresses it.

This bucket is the most clinically meaningful because it represents cases in which degradation actually hides information that should be detected.

In this key bucket (**N = 36**), V-Ultimate achieved:

- **mean_target_gain = 0.0485** (**highest**)
- **mean_abs_gain = 0.0664** (**highest**)
- **iatrogenic = 16.7%** (**lowest**)
- **psnr = 42.56 dB** (**highest**)

By comparison:

- Gaussian: target_gain = **0.0194**, iatrogenic = **36.1%**
- Bilateral: target_gain = **0.0206**, iatrogenic = **22.2%**
- Median3: target_gain = **0.0082**, iatrogenic = **38.9%**

This is the central result of the CRM experiment:

> **When degradation truly damages clinically relevant signal, V-Ultimate is the best method at rescuing that signal, and it does so with the lowest harm rate.**

That is much more meaningful than simply saying the model “denoises better.”

---

### 5. This suggests the model is learning targeted recovery, not blind smoothing

If a method is merely smoothing, two patterns often appear:

1. it may help in some clearly degraded cases;
2. but it also tends to over-edit or over-smooth cases that do not need much intervention.

V-Ultimate’s gains are concentrated most strongly in the **`harmed_positive`** bucket, which is exactly where recovery is most needed.

That matches the intended design of the method:

> **The model is not trying to smooth every image uniformly; it is more effective when signal has actually been damaged and needs rescue.**

This makes V-Ultimate better described as a **target-oriented restoration model** rather than just a generic denoiser.

---

### 6. Important limitation: neutral and overcall cases remain challenging

The results also show clearly that V-Ultimate is **not perfect**.

In the **`neutral_other`** bucket (**N = 12**):

- mean_target_gain = **-0.0123**
- iatrogenic = **75.0%**

This suggests that:

> when degradation does not strongly suppress the target signal, the model can still introduce unnecessary edits.

In the **`overcall_positive`** bucket (**N = 2**), the sample size is too small for strong conclusions, but the current result also indicates instability in cases where the task is to reduce inflated signal rather than recover suppressed signal.

Therefore, the correct conclusion is **not**:

> “V-Ultimate is best in every type of case.”

Instead, it is:

> **V-Ultimate has a very clear strength profile: it is most useful in the clinically important cases where degradation truly harms the target, while still leaving room for improvement in cases that may not require substantial correction.**

This is exactly why the later Monte Carlo, Mayo, and anatomy-aware analyses are important.

---

### 7. Pairwise comparison further supports that the advantage is systematic, not driven by a few lucky cases

Using V-Ultimate as the reference in paired comparisons:

- vs Gaussian:  
  - **V wins TGain % = 64.0%**
  - ΔTGain = **+0.0199**

- vs Median3:  
  - **V wins TGain % = 74.0%**
  - ΔTGain = **+0.0288**

- vs Bilateral:  
  - **V wins TGain % = 62.0%**
  - ΔTGain = **+0.0209**

- vs Unsharp:  
  - **V wins TGain % = 82.0%**
  - ΔTGain = **+0.0499**

This shows that V-Ultimate’s advantage is not driven by a handful of outliers. Instead, it wins in the majority of matched case-level comparisons, including against strong traditional baselines such as Gaussian and Bilateral.

---

### 8. Final interpretation of this section

The CRM results support not a simple “score-chasing” conclusion, but a more important methodological one:

> **A controlled restoration model trained only with physics-based synthetic supervision can already produce real, systematic recovery benefits on strict held-out OOD clinical proxy tasks.**

More specifically:

1. **It achieves the highest overall target gain and the highest repair win rate;**
2. **Its strongest advantage appears in the clinically important cases where degradation actually suppresses signal;**
3. **Its behavior cannot be explained by generic smoothing alone, because its gains are concentrated in rescue-relevant cases;**
4. **At the same time, it still struggles in neutral or overcall settings, which motivates the later stability analysis and external real-data validation.**

Therefore, the most accurate conclusion is not:

> “V-Ultimate completely crushes all traditional baselines.”

It is:

> **V-Ultimate demonstrates that physics-based synthetic supervision can train a controlled restoration model that meaningfully rescues clinically relevant signal under strict OOD conditions.**

In [ ]:
# ============================================================
# 8) Stage C: Monte Carlo Noise Stability Test (NEW MODEL)
# Compatible with: FiLM + authority map V-Ultimate
# ============================================================

import os, sys, time, math, random, gc
import numpy as np
import pandas as pd
import cv2
import torch
from contextlib import nullcontext

try:
    from IPython.display import display
except Exception:
    display = print

try:
    cv2.setNumThreads(0)
except Exception:
    pass

# -----------------------------
# 0) Prerequisites check
# -----------------------------
required_any = {
    "model": ("model_25d" in globals()),
    "judge": ("aneurysm_predict" in globals()),
    "loader": ("load_series_volume" in globals()),
}
if not all(required_any.values()):
    missing = [k for k, v in required_any.items() if not v]
    raise RuntimeError(
        f"Missing prerequisite objects/functions: {missing}\n"
        f"Please run: new model loading + Clinical Judge + DICOM loading cells first."
    )

MODEL_OBJ = globals().get("model_25d", None)
assert MODEL_OBJ is not None, "model_25d not found"
assert hasattr(MODEL_OBJ, "auth_head"), "Not the new model: missing auth_head"
assert hasattr(MODEL_OBJ, "film_e1"), "Not the new model: missing FiLM module"

# -----------------------------
# 1) Configuration
# -----------------------------
RSNA_DATA_ROOT = globals().get("RSNA_DATA_ROOT", "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/series")
META_CSV       = globals().get("META_CSV", "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv")

TARGET_D = int(globals().get("TARGET_D", 64))
TARGET_H = int(globals().get("TARGET_H", 448))
TARGET_W = int(globals().get("TARGET_W", 448))
TARGET_SHAPE = (TARGET_D, TARGET_H, TARGET_W)

HU_MIN   = float(globals().get("HU_MIN", -1024.0))
HU_MAX   = float(globals().get("HU_MAX", 3072.0))
HU_RANGE = float(globals().get("HU_RANGE", HU_MAX - HU_MIN))

DIFFUSION_ALPHA = float(globals().get("DIFFUSION_ALPHA", globals().get("LAM", 0.20)))
BLUR_T_MAX = float(globals().get("BLUR_T_MAX", 8.0))

EVAL_T    = float(globals().get("EVAL_T", 8.0))
EVAL_DOSE = globals().get("EVAL_DOSE", "quarter")

N_MC_TO_RUN = int(globals().get("N_MC_CASES", 100))
MC_SEEDS    = list(globals().get("MC_SEEDS", list(range(10))))
RESTORE_BATCH = int(globals().get("RESTORE_BATCH", 16))

from pathlib import Path
OUTDIR = str(Path(CRM_OUTDIR) / "monte_carlo_stability")
os.makedirs(OUTDIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (device.type == "cuda")
AMP_CTX = (lambda: torch.amp.autocast("cuda")) if USE_AMP else (lambda: nullcontext())

# Thresholds
TARGET_POS_TH = 0.005
TARGET_NEG_TH = -0.005
ABS_POS_TH = 0.005
ABS_NEG_TH = -0.005

# -----------------------------
# 2) Utility functions
# -----------------------------
def mc_bucket(p_gt, p_deg, tau=0.03):
    if p_gt >= 0.5 and p_deg < p_gt - tau:
        return "harmed_positive"
    if p_gt >= 0.5 and p_deg > p_gt + tau:
        return "overcall_positive"
    return "neutral_other"

def degrade_volume_mc_seeded(vol01, t, dose_mode="quarter", enable_motion=False):
    """
    Stochastic degradation matching training/CRM style.
    Randomness is controlled by the external seed set before calling.
    """
    D = vol01.shape[0]
    out = np.empty_like(vol01, dtype=np.float32)

    sigma = math.sqrt(max(1e-8, 2.0 * DIFFUSION_ALPHA * float(t)))

    for z in range(D):
        x = vol01[z].astype(np.float32)
        x = cv2.GaussianBlur(
            x, (0, 0),
            sigmaX=sigma, sigmaY=sigma,
            borderType=cv2.BORDER_REPLICATE
        )

        # Motion artefact (disabled by default)
        if enable_motion:
            pass

        if dose_mode != "clean":
            if dose_mode == "extreme":
                peak = random.uniform(1000.0, 3000.0)
                sigma_e = random.uniform(0.02, 0.04)
            else:
                peak = random.uniform(3000.0, 6000.0)
                sigma_e = random.uniform(0.01, 0.02)

            noisy_p = np.random.poisson(np.clip(x * peak, 0, None)).astype(np.float32) / peak
            x = noisy_p + np.random.randn(*x.shape).astype(np.float32) * sigma_e

        out[z] = np.clip(x, 0.0, 1.0).astype(np.float32)

    return out

@torch.no_grad()
def run_vultimate_mc(vol_deg01, t=EVAL_T, restore_batch=RESTORE_BATCH):
    """
    FiLM + authority map compatible inference.
    meta = [t_norm, do_motion, dose_clean, is_identity]
    """
    vol_deg01 = np.asarray(vol_deg01, dtype=np.float32)
    D = vol_deg01.shape[0]
    out = vol_deg01.copy()

    t_norm = np.float32(0.0 if t <= 0 else (float(t) / float(BLUR_T_MAX)))
    meta_row = np.array([t_norm, 0.0, 0.0, 0.0], dtype=np.float32)

    for s in range(0, D, restore_batch):
        zs = list(range(s, min(D, s + restore_batch)))
        inp_batch = []

        for z in zs:
            bp = vol_deg01[max(0, z - 1)]
            bc = vol_deg01[z]
            bn = vol_deg01[min(D - 1, z + 1)]
            inp_batch.append(
                np.stack([bp, bc, bn, np.full_like(bc, t_norm)], axis=0).astype(np.float32)
            )

        inp_t = torch.from_numpy(np.stack(inp_batch, axis=0)).to(device, non_blocking=True)
        meta_t = torch.from_numpy(np.repeat(meta_row[None, :], len(zs), axis=0)).to(device, non_blocking=True)

        with AMP_CTX():
            pred_obj = MODEL_OBJ(inp_t, meta=meta_t)
            if isinstance(pred_obj, (tuple, list)):
                pred_b = pred_obj[0].float().cpu().numpy()[:, 0]
            else:
                pred_b = pred_obj.float().cpu().numpy()[:, 0]

        for k, z in enumerate(zs):
            out[z] = np.clip(pred_b[k], 0.0, 1.0).astype(np.float32)

    return out

# -----------------------------
# 3) Build OOD CT/CTA pool
# -----------------------------
print("=== Stage C | Monte Carlo Stability Test ===")

meta = pd.read_csv(META_CSV)
ct_uids = set(meta[meta["Modality"].astype(str).isin({"CT", "CTA"})]["SeriesInstanceUID"].astype(str).tolist())

train_uid_set = set()
for p in [
    "/kaggle/working/vultimate_sleep_safe/train_uids_ultimate.csv",
    "/kaggle/working/train_uids_ultimate.csv",
    "/kaggle/input/datasets/mingzeli2009/train-uids-ultimate/train_uids_ultimate.csv",
    "/kaggle/working/train_uids_ct_only.csv",
]:
    if os.path.exists(p):
        try:
            train_uid_set = set(pd.read_csv(p)["SeriesInstanceUID"].astype(str).tolist())
            print(f"[Train exclusion] loaded {len(train_uid_set)} train UIDs from: {p}")
            break
        except Exception:
            pass

all_series_dirs = [u for u in os.listdir(RSNA_DATA_ROOT) if os.path.isdir(os.path.join(RSNA_DATA_ROOT, u))]
ood_pool = [u for u in all_series_dirs if (u in ct_uids) and (u not in dev_uid_set)]

random.Random(2026).shuffle(ood_pool)

mc_candidates = ood_pool[:max(N_MC_TO_RUN * 3, 300)]

# -----------------------------
# 4) Pre-load cases
# -----------------------------
mc_cases = []
t_prep = time.time()

for uid in mc_candidates:
    if len(mc_cases) >= N_MC_TO_RUN:
        break

    try:
        vol = load_series_volume(uid, RSNA_DATA_ROOT, TARGET_SHAPE)
    except TypeError:
        try:
            vol = load_series_volume(uid, RSNA_DATA_ROOT)
        except TypeError:
            vol = load_series_volume(uid)

    if vol is None:
        continue

    gt_vol = vol.astype(np.float32)
    p_gt = float(aneurysm_predict(vol01_to_flayer_uint8(gt_vol)))

    mc_cases.append({
        "uid": uid,
        "uid4": uid_tail4(uid),
        "gt_vol": gt_vol,
        "p_gt": p_gt,
    })

    if (len(mc_cases) == 1) or (len(mc_cases) % 10 == 0) or (len(mc_cases) == N_MC_TO_RUN):
        print(f"[prep {len(mc_cases):03d}/{N_MC_TO_RUN}] UID:{uid_tail4(uid)} | p_gt={p_gt:.4f}")

print(f"MC selected cases: {len(mc_cases)} / {N_MC_TO_RUN}")
print(f"MC seeds: {MC_SEEDS}")
print(f"MC preparation elapsed: {(time.time()-t_prep)/60:.1f} min")

# -----------------------------
# 5) Monte Carlo main loop
# -----------------------------
mc_raw_rows = []
t_mc = time.time()

for ci, case in enumerate(mc_cases, 1):
    uid = case["uid"]
    uid4 = case["uid4"]
    gt_vol = case["gt_vol"]
    p_gt = float(case["p_gt"])

    print(f"\n[{ci:03d}/{len(mc_cases)}] UID:{uid4} | p_gt={p_gt:.4f} | running {len(MC_SEEDS)} seeds ...")
    case_t0 = time.time()

    case_bucket_votes = []

    for s in MC_SEEDS:
        random.seed(s)
        np.random.seed(s)
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)

        deg_vol = degrade_volume_mc_seeded(gt_vol, EVAL_T, dose_mode=EVAL_DOSE, enable_motion=False)
        rec_vol = run_vultimate_mc(deg_vol, t=EVAL_T, restore_batch=RESTORE_BATCH)

        p_deg = float(aneurysm_predict(vol01_to_flayer_uint8(deg_vol)))
        p_rec = float(aneurysm_predict(vol01_to_flayer_uint8(rec_vol)))

        bucket = mc_bucket(p_gt, p_deg, tau=0.03)
        case_bucket_votes.append(bucket)

        t_gain = float(calc_target_gain(p_gt, p_deg, p_rec))
        a_gain = float(calc_abs_gain(p_gt, p_deg, p_rec))

        mc_raw_rows.append({
            "uid_full": uid,
            "uid4": uid4,
            "seed": int(s),
            "bucket": bucket,
            "p_gt": p_gt,
            "p_deg": p_deg,
            "p_rec": p_rec,
            "target_gain": t_gain,
            "abs_gain": a_gain,
            "target_positive": int(t_gain > TARGET_POS_TH),
            "target_negative": int(t_gain < TARGET_NEG_TH),
            "abs_positive": int(a_gain > ABS_POS_TH),
            "abs_negative": int(a_gain < ABS_NEG_TH),
        })

        del deg_vol, rec_vol
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tmp = pd.DataFrame([r for r in mc_raw_rows if r["uid_full"] == uid])
    major_bucket = tmp["bucket"].mode().iloc[0] if len(tmp) else "NA"

    print(
        "  -> case done in {:.1f}s | bucket={} | "
        "TGain mean={:+.4f}, std={:.4f}, pos/neg={:.2f}/{:.2f} | "
        "AGain mean={:+.4f}, pos/neg={:.2f}/{:.2f}".format(
            time.time()-case_t0,
            major_bucket,
            tmp["target_gain"].mean(), tmp["target_gain"].std(ddof=0),
            (tmp["target_gain"] > TARGET_POS_TH).mean(), (tmp["target_gain"] < TARGET_NEG_TH).mean(),
            tmp["abs_gain"].mean(),
            (tmp["abs_gain"] > ABS_POS_TH).mean(), (tmp["abs_gain"] < ABS_NEG_TH).mean(),
        )
    )

df_mc_raw = pd.DataFrame(mc_raw_rows)

# -----------------------------
# 6) Aggregation
# -----------------------------
def agg_mc_case(g):
    t = g["target_gain"].to_numpy(dtype=float)
    a = g["abs_gain"].to_numpy(dtype=float)
    pdeg = g["p_deg"].to_numpy(dtype=float)
    prec = g["p_rec"].to_numpy(dtype=float)
    pgt = float(g["p_gt"].iloc[0])

    t_pos = t > TARGET_POS_TH
    t_neg = t < TARGET_NEG_TH
    a_pos = a > ABS_POS_TH
    a_neg = a < ABS_NEG_TH

    bucket_mode = g["bucket"].mode().iloc[0] if len(g["bucket"].mode()) else "NA"

    return pd.Series({
        "n_runs": int(len(g)),
        "bucket_major": bucket_mode,
        "p_gt": pgt,

        "target_gain_mean": float(np.mean(t)),
        "target_gain_std": float(np.std(t, ddof=0)),
        "target_gain_min": float(np.min(t)),
        "target_gain_max": float(np.max(t)),
        "target_pos_rate": float(np.mean(t_pos)),
        "target_neg_rate": float(np.mean(t_neg)),
        "target_flip": bool(np.any(t_pos) and np.any(t_neg)),

        "abs_gain_mean": float(np.mean(a)),
        "abs_gain_std": float(np.std(a, ddof=0)),
        "abs_gain_min": float(np.min(a)),
        "abs_gain_max": float(np.max(a)),
        "abs_pos_rate": float(np.mean(a_pos)),
        "abs_neg_rate": float(np.mean(a_neg)),
        "abs_flip": bool(np.any(a_pos) and np.any(a_neg)),

        "p_deg_std": float(np.std(pdeg, ddof=0)),
        "p_rec_std": float(np.std(prec, ddof=0)),
    })

df_mc_agg = (
    df_mc_raw.groupby(["uid_full", "uid4"], as_index=False)
    .apply(agg_mc_case)
    .reset_index(drop=True)
)

mc_raw_path = os.path.join(OUTDIR, "mc_noise_100cases_10seeds_raw.csv")
mc_agg_path = os.path.join(OUTDIR, "mc_noise_100cases_10seeds_agg.csv")
df_mc_raw.to_csv(mc_raw_path, index=False)
df_mc_agg.to_csv(mc_agg_path, index=False)

# -----------------------------
# 7) Summary statistics
# -----------------------------
n = len(df_mc_agg)

stable_pos = ((df_mc_agg["target_pos_rate"] > 0) & (df_mc_agg["target_neg_rate"] == 0)).sum()
flip       = ((df_mc_agg["target_pos_rate"] > 0) & (df_mc_agg["target_neg_rate"] > 0)).sum()
stable_neg = ((df_mc_agg["target_pos_rate"] == 0) & (df_mc_agg["target_neg_rate"] > 0)).sum()
neutral    = ((df_mc_agg["target_pos_rate"] == 0) & (df_mc_agg["target_neg_rate"] == 0)).sum()

mc_summary = {
    "n_cases": n,
    "n_seeds_per_case": len(MC_SEEDS),
    "n_total_runs": len(df_mc_raw),

    "target_gain_mean(run-level)": float(df_mc_raw["target_gain"].mean()),
    "target_positive_rate(run-level)": float((df_mc_raw["target_gain"] > TARGET_POS_TH).mean()),
    "target_negative_rate(run-level)": float((df_mc_raw["target_gain"] < TARGET_NEG_TH).mean()),

    "abs_gain_mean(run-level)": float(df_mc_raw["abs_gain"].mean()),

    "cases_with_target_flip": int(df_mc_agg["target_flip"].sum()),
    "stable_positive": int(stable_pos),
    "noise_sensitive_flip": int(flip),
    "stable_negative": int(stable_neg),
    "neutral": int(neutral),

    "elapsed_min": round((time.time() - t_mc) / 60.0, 1),
}

print("\n" + "="*100)
print(f"Monte Carlo Stability Summary ({n} cases × {len(MC_SEEDS)} seeds)")
print("="*100)
for k, v in mc_summary.items():
    if isinstance(v, float):
        print(f"{k:>45}: {v:.4f}")
    else:
        print(f"{k:>45}: {v}")

print(f"\nCase classification:")
print(f"   Stable positive:    {stable_pos}/{n} ({stable_pos/n*100:.0f}%)")
print(f"   Noise-sensitive:    {flip}/{n} ({flip/n*100:.0f}%)")
print(f"   Stable negative:    {stable_neg}/{n} ({stable_neg/n*100:.0f}%)")
print(f"   Neutral:            {neutral}/{n} ({neutral/n*100:.0f}%)")

print("\n[Top noise-sensitive cases by target_gain_std]")
display(
    df_mc_agg.sort_values(["target_gain_std", "target_neg_rate"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)

print("\n[Top unstable / failure-prone cases by target_neg_rate]")
display(
    df_mc_agg.sort_values(["target_neg_rate", "target_gain_std"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)

print("\n[Bucket breakdown]")
display(
    df_mc_agg.groupby("bucket_major", as_index=False)
    .agg(
        n_cases=("uid_full", "size"),
        mean_target_gain=("target_gain_mean", "mean"),
        mean_target_std=("target_gain_std", "mean"),
        cases_with_flip=("target_flip", "sum"),
        mean_prec_std=("p_rec_std", "mean"),
    )
)

print("\nsaved:", mc_raw_path)
print("saved:", mc_agg_path)
print("\n✅ Monte Carlo complete.")


## Results Interpretation — Monte Carlo Stability Test

### 1. What is this experiment testing?

The Monte Carlo stability test asks a different question from the main CRM table.

The CRM summary tells us whether V-Ultimate is beneficial **on average** across a fixed evaluation set.  
The Monte Carlo analysis asks:

> **If the degradation noise realization changes while the same underlying case is kept fixed, does the model still behave consistently?**

This is important because a restoration method can look strong in one deterministic evaluation, yet be unstable when the exact corruption pattern varies.

In other words, this section measures **robustness to stochastic degradation**, not just average performance.

---

### 2. Overall result: the method remains net-positive under repeated stochastic perturbations

Across **100 cases × 10 noise seeds = 1000 total runs**, the run-level summary is:

- **Mean target gain = 0.0335**
- **Target-positive rate = 71.7%**
- **Target-negative rate = 20.3%**
- **Mean absolute gain = 0.0549**

This is the most important first conclusion:

> **Even after repeatedly perturbing each case with different stochastic degradation realizations, V-Ultimate remains net-positive overall.**

So the main CRM conclusion is **not** a one-seed accident.  
The model still shows a positive average rescue effect under repeated noisy corruption.

---

### 3. But the stability is not uniform: most cases are noise-sensitive to some degree

The case-level stability breakdown is:

- **Stable positive:** 36 / 100 (**36%**)
- **Noise-sensitive:** 62 / 100 (**62%**)
- **Stable negative:** 2 / 100 (**2%**)
- **Neutral:** 0 / 100 (**0%**)

This means the correct interpretation is **not**:

> “The model behaves identically under all noise realizations.”

Instead, the correct interpretation is:

> **The model is beneficial overall, but its case-level response is often sensitive to the exact stochastic corruption pattern.**

That is why this section is so important. It shows that the method is **robust in expectation**, but **not uniformly deterministic at the individual-case level**.

---

### 4. The most encouraging result: clinically important harmed-positive cases remain positive on average

The most important bucket is still **`harmed_positive`**, where degradation suppresses a clinically relevant signal that should ideally be recovered.

For this bucket:

- **n_cases = 80**
- **mean_target_gain = 0.0442**
- **mean_target_std = 0.0324**
- **cases_with_flip = 45**
- **mean_pred_std = 0.0243**

This is the core takeaway:

> **In the clinically important harmed-positive cases, the average effect remains clearly positive, even though many individual cases still exhibit seed-to-seed fluctuation.**

So the model is doing the right thing in the most relevant bucket **on average**, but some cases sit close enough to the decision boundary that different noise realizations can change whether the outcome is counted as positive or negative.

That is exactly why many cases are labeled **noise-sensitive**, rather than simply “successful” or “failed.”

---

### 5. Neutral and overcall cases remain the main weakness

The weaker stability appears in the buckets where recovery is less clearly needed:

#### `neutral_other`
- **n_cases = 18**
- **mean_target_gain = -0.0093**
- **mean_target_std = 0.0271**
- **cases_with_flip = 15**

#### `overcall_positive`
- **n_cases = 2**
- **mean_target_gain = -0.0126**
- **mean_target_std = 0.0309**
- **cases_with_flip = 2**

These results are consistent with the earlier CRM interpretation:

> **V-Ultimate is strongest when the target signal has genuinely been damaged, but it is less reliable when the case is neutral or already overcalled.**

This matters because it shows that the model’s current limitation is not random across all cases.  
Its instability is concentrated in the scenarios where the restoration objective is inherently more ambiguous.

---

### 6. The meaning of “noise-sensitive” is important

A “noise-sensitive” case does **not** necessarily mean the model is bad.  
It means that across different stochastic corruptions of the same underlying case, the direction or strength of the rescue effect can change.

For example:

- some cases remain positive, but with variable strength;
- some cases occasionally cross from positive to negative;
- some cases are near the clinical decision boundary and are therefore especially sensitive.

So the right interpretation is:

> **Noise-sensitive cases are not all failures; they are cases where restoration outcome depends meaningfully on the exact degradation realization.**

This is a more nuanced and scientifically useful conclusion than simply labeling everything as success or failure.

---

### 7. The top unstable cases show where the method still needs improvement

The most unstable examples are informative because they reveal the current edge cases of the method.

Two patterns stand out:

#### A. Some `harmed_positive` cases are beneficial on average, but still flip across seeds
Examples such as `3439`, `0800`, `4778`, and `6455` have:

- positive mean target gain,
- but substantial standard deviation,
- and seed-level sign flips.

This suggests that:

> the model often moves in the right direction, but its response is still somewhat sensitive when the corruption severity lands near a difficult boundary.

#### B. Many `neutral_other` cases are consistently fragile
Examples such as `6332`, `3779`, `3477`, and `3141` show:

- negative or near-zero mean target gain,
- frequent flips,
- and clear instability across seeds.

This reinforces the earlier finding that:

> **the model is currently much better at rescuing damaged signal than at deciding when to leave an already acceptable case alone.**

---

### 8. What does this mean for the overall claim of the project?

The Monte Carlo result does **not** weaken the project’s central claim.  
Instead, it sharpens it.

A naive interpretation would be:

> “Because many cases are noise-sensitive, the method is unreliable.”

That would be too strong.

A better interpretation is:

> **The method is robust at the population level and clearly beneficial in the most clinically relevant harmed-positive bucket, but it is not yet fully stable at the individual-case level under stochastic degradation variation.**

This is actually a meaningful scientific result, because it shows both:

1. **why the method is promising**, and  
2. **where its current limits still are.**

---

### 9. Final interpretation of this section

The Monte Carlo stability test supports the following conclusion:

> **V-Ultimate’s CRM benefit is not a single-seed artifact: under repeated stochastic degradation, the method remains net-positive overall and especially positive in clinically important harmed-positive cases.**

At the same time, it also reveals an important limitation:

> **Case-level behavior is often noise-sensitive, particularly in neutral or ambiguous cases, indicating that the model still needs stronger mechanisms for deciding when not to edit.**

Therefore, this section strengthens the project in two ways:

- it confirms that the observed rescue effect is **real and reproducible in expectation**;
- it identifies **precisely where further safety and stability improvements are needed**.

The correct summary is not:

> “The model is perfectly stable.”

It is:

> **The model is directionally stable at the population level, strongest where clinically relevant signal is genuinely suppressed, but still sensitive in ambiguous cases under stochastic corruption.**

In [ ]:
# ============================================================
# Mayo external paired evaluation — 10 unique matched series
# - strict matching by relative directory
# - exactly one center window per matched series
# - no repeated windows from the same series
# Depends on:
#   - model_25d, device, AMP_CTX
#   - Q_DIR, F_DIR, MAYO_OUTDIR
# ============================================================

MAYO_ROOT = "/kaggle/input/datasets/andrewmvd/ct-low-dose-reconstruction/CT_low_dose_reconstruction_dataset/Original Data"
Q_DIR = os.path.join(MAYO_ROOT, "Quarter Dose")
F_DIR = os.path.join(MAYO_ROOT, "Full Dose")

MAYO_OUTDIR = Path("/kaggle/working/vultimate_scientific_eval/mayo_external_eval")
MAYO_OUTDIR.mkdir(parents=True, exist_ok=True)

print("MAYO_ROOT exists?", os.path.exists(MAYO_ROOT))
print("Q_DIR exists?", os.path.exists(Q_DIR))
print("F_DIR exists?", os.path.exists(F_DIR))
print("MAYO_OUTDIR:", MAYO_OUTDIR)

import os, gc, math, time
from pathlib import Path
from contextlib import nullcontext

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# 0) Config / checks
# ------------------------------------------------------------
required_globals = [
    "model_25d", "device", "AMP_CTX",
    "Q_DIR", "F_DIR", "MAYO_OUTDIR",
    "HU_MIN", "HU_MAX", "HU_RANGE",
    "BLUR_T_MAX", "RESTORE_BATCH",
]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals: {missing}")

OUTDIR = Path(MAYO_OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)

N_CASES_MAIN = int(globals().get("N_CASES_MAIN", 10))   # unique matched series
VOL_DEPTH = int(globals().get("VOL_DEPTH", 128))
TARGET_H = int(globals().get("TARGET_H", 448))
TARGET_W = int(globals().get("TARGET_W", 448))

T_INFER_NORM = 0.05
T_INFER = T_INFER_NORM * float(BLUR_T_MAX)
MAYO_USE_GAUSSIAN = True
MAYO_GAUSSIAN_SIGMA = 0.8
BOOT_N = 2000
SEED_MAYO = 2026

print("Q_DIR exists?", os.path.exists(Q_DIR))
print("F_DIR exists?", os.path.exists(F_DIR))
print("N_CASES_MAIN:", N_CASES_MAIN)

# ------------------------------------------------------------
# 1) Helpers
# ------------------------------------------------------------
def is_dicom_like(fname: str):
    if fname.startswith("."):
        return False
    if fname.endswith((".dcm", ".DCM", ".ima", ".IMA")):
        return True
    if "." not in fname:
        return True
    return False

def dcm_sort_key(fp: str):
    try:
        ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)

        inst = getattr(ds, "InstanceNumber", None)
        if inst is not None:
            try:
                return (0, int(inst), fp)
            except Exception:
                pass

        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp is not None and len(ipp) >= 3:
            try:
                return (1, float(ipp[2]), fp)
            except Exception:
                pass

        sl = getattr(ds, "SliceLocation", None)
        if sl is not None:
            try:
                return (2, float(sl), fp)
            except Exception:
                pass
    except Exception:
        pass

    return (9, fp)

def list_dicom_series(root_dir):
    series = {}
    for r, _, fs in os.walk(root_dir):
        dcm_files = [os.path.join(r, f) for f in fs if is_dicom_like(f)]
        if len(dcm_files) == 0:
            continue
        rel = os.path.relpath(r, root_dir)
        series[rel] = sorted(dcm_files, key=dcm_sort_key)
    return series

def dcm_to_hu(ds):
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    return arr * slope + intercept

def hu_to_01(hu):
    return np.clip((hu - HU_MIN) / HU_RANGE, 0.0, 1.0).astype(np.float32)

def hu01_to_hu(x01):
    return np.asarray(x01, dtype=np.float32) * HU_RANGE + HU_MIN

def psnr01(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    mse = float(np.mean((a - b) ** 2))
    return 99.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse)

def mae01(a, b):
    return float(np.mean(np.abs(np.asarray(a, dtype=np.float32) - np.asarray(b, dtype=np.float32))))

def ssim_fast01_2d(a, b):
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu_a = cv2.GaussianBlur(a, (11, 11), 1.5)
    mu_b = cv2.GaussianBlur(b, (11, 11), 1.5)
    mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
    sigma_a2 = cv2.GaussianBlur(a * a, (11, 11), 1.5) - mu_a2
    sigma_b2 = cv2.GaussianBlur(b * b, (11, 11), 1.5) - mu_b2
    sigma_ab = cv2.GaussianBlur(a * b, (11, 11), 1.5) - mu_ab
    ssim_map = ((2 * mu_ab + C1) * (2 * sigma_ab + C2)) / (
        (mu_a2 + mu_b2 + C1) * (sigma_a2 + sigma_b2 + C2) + 1e-8
    )
    return float(np.mean(ssim_map))

def build_paired_case_manifest_unique(q_root, f_root, depth=128, n_target=10):
    q_series = list_dicom_series(q_root)
    f_series = list_dicom_series(f_root)
    common_rels = sorted(set(q_series.keys()) & set(f_series.keys()))

    cases = []
    for rel in common_rels:
        qf = q_series[rel]
        ff = f_series[rel]
        usable_len = min(len(qf), len(ff))
        if usable_len < depth:
            continue

        center_start = max(0, (usable_len - depth) // 2)
        cases.append({
            "rel_dir": rel,
            "start_idx": int(center_start),
            "usable_len": int(usable_len),
            "q_files": qf,
            "f_files": ff,
        })

    if len(cases) == 0:
        raise RuntimeError("No matched Quarter/Full series with enough slices were found.")

    # prefer longer usable series; one window per unique matched series only
    cases = sorted(cases, key=lambda x: (-x["usable_len"], x["rel_dir"]))[:n_target]

    out = []
    for i, m in enumerate(cases, 1):
        out.append({
            "case_id": i,
            "rel_dir": m["rel_dir"],
            "start_idx": m["start_idx"],
            "usable_len": m["usable_len"],
            "q_files": m["q_files"],
            "f_files": m["f_files"],
        })
    return out

def load_paired_case_window(q_files, f_files, start_idx, depth=128, out_hw=(448, 448)):
    q_hu_list, f_hu_list = [], []
    max_len = min(len(q_files), len(f_files))

    for k in range(depth):
        idx = start_idx + k
        if idx < 0 or idx >= max_len:
            break

        ds_q = pydicom.dcmread(q_files[idx], force=True)
        ds_f = pydicom.dcmread(f_files[idx], force=True)

        hq = dcm_to_hu(ds_q)
        hf = dcm_to_hu(ds_f)

        if hq.shape != out_hw:
            hq = cv2.resize(hq, (out_hw[1], out_hw[0]), interpolation=cv2.INTER_LINEAR)
        if hf.shape != out_hw:
            hf = cv2.resize(hf, (out_hw[1], out_hw[0]), interpolation=cv2.INTER_LINEAR)

        q_hu_list.append(hq.astype(np.float32))
        f_hu_list.append(hf.astype(np.float32))

    q_hu = np.stack(q_hu_list, axis=0).astype(np.float32)
    f_hu = np.stack(f_hu_list, axis=0).astype(np.float32)
    q01 = hu_to_01(q_hu)
    f01 = hu_to_01(f_hu)
    return q_hu, f_hu, q01, f01

@torch.no_grad()
def run_vultimate_mayo(vol_deg01, t=T_INFER, restore_batch=RESTORE_BATCH):
    vol_deg01 = np.asarray(vol_deg01, dtype=np.float32)
    D = vol_deg01.shape[0]
    out = vol_deg01.copy()

    t_norm = np.float32(0.0 if t <= 0 else float(t) / float(BLUR_T_MAX))
    meta_row = np.array([t_norm, 0.0, 0.0, 0.0], dtype=np.float32)

    for s in range(0, D, restore_batch):
        zs = list(range(s, min(D, s + restore_batch)))
        inp_batch = []

        for z in zs:
            bp = vol_deg01[max(0, z - 1)]
            bc = vol_deg01[z]
            bn = vol_deg01[min(D - 1, z + 1)]
            inp_batch.append(np.stack([bp, bc, bn, np.full_like(bc, t_norm)], axis=0).astype(np.float32))

        inp_t = torch.from_numpy(np.stack(inp_batch, axis=0)).to(device, non_blocking=True)
        meta_t = torch.from_numpy(np.repeat(meta_row[None, :], len(zs), axis=0)).to(device, non_blocking=True)

        with AMP_CTX():
            pred_obj = model_25d(inp_t, meta=meta_t)
            pred = pred_obj[0] if isinstance(pred_obj, (tuple, list)) else pred_obj
            pred = pred.float().cpu().numpy()[:, 0]

        for k, z in enumerate(zs):
            out[z] = np.clip(pred[k], 0.0, 1.0).astype(np.float32)

    return out

def gaussian_baseline_3d(vol01, sigma=0.8):
    out = np.empty_like(vol01, dtype=np.float32)
    for z in range(vol01.shape[0]):
        out[z] = cv2.GaussianBlur(
            vol01[z], (0, 0), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REPLICATE
        )
    return np.clip(out, 0.0, 1.0).astype(np.float32)

def vol_metrics(x01, ref01, q01_for_change):
    z_idx = list(range(0, x01.shape[0], 8))
    ssim_vals = [ssim_fast01_2d(x01[z], ref01[z]) for z in z_idx]
    return {
        "psnr": psnr01(x01, ref01),
        "mae": mae01(x01, ref01),
        "ssim": float(np.mean(ssim_vals)),
        "mean_abs_change_vs_quarter": float(np.mean(np.abs(x01 - q01_for_change))),
        "max_abs_change_vs_quarter": float(np.max(np.abs(x01 - q01_for_change))),
    }

def paired_bootstrap_ci(x, y, n_boot=2000, seed=2026):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    d = x - y
    mean_d = float(np.mean(d))

    rng = np.random.default_rng(seed)
    boots = []
    n = len(d)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots.append(float(np.mean(d[idx])))

    lo = float(np.percentile(boots, 2.5))
    hi = float(np.percentile(boots, 97.5))
    return mean_d, lo, hi

# ------------------------------------------------------------
# 2) Build strict unique-series manifest
# ------------------------------------------------------------
mayo_case_manifest = build_paired_case_manifest_unique(
    Q_DIR, F_DIR, depth=VOL_DEPTH, n_target=N_CASES_MAIN
)

df_manifest = pd.DataFrame([
    {
        "case_id": m["case_id"],
        "rel_dir": m["rel_dir"],
        "start_idx": m["start_idx"],
        "usable_len": m["usable_len"],
        "q_nfiles": len(m["q_files"]),
        "f_nfiles": len(m["f_files"]),
    }
    for m in mayo_case_manifest
])

assert df_manifest["rel_dir"].nunique() == len(df_manifest), "Expected unique matched series only."

df_manifest.to_csv(OUTDIR / "mayo_case_manifest.csv", index=False)

print("\n=== Mayo case manifest ===")
display(df_manifest)
print("Total selected windows:", len(df_manifest))
print("Unique matched series :", df_manifest["rel_dir"].nunique())

# ------------------------------------------------------------
# 3) Main experiment
# ------------------------------------------------------------
rows = []
t0 = time.time()

for i, case_info in enumerate(mayo_case_manifest, 1):
    cid = int(case_info["case_id"])
    rel_dir = case_info["rel_dir"]
    st = int(case_info["start_idx"])

    print(f"\n[{i}/{len(mayo_case_manifest)}] case_id={cid:02d} | rel_dir={rel_dir} | start={st}")

    q_hu, f_hu, q01, f01 = load_paired_case_window(
        case_info["q_files"], case_info["f_files"], st, depth=VOL_DEPTH, out_hw=(TARGET_H, TARGET_W)
    )

    m_q = vol_metrics(q01, f01, q01)
    rows.append({
        "case_id": cid, "rel_dir": rel_dir, "start_idx": st, "variant": "Quarter",
        **m_q,
    })

    if MAYO_USE_GAUSSIAN:
        g01 = gaussian_baseline_3d(q01, sigma=MAYO_GAUSSIAN_SIGMA)
        m_g = vol_metrics(g01, f01, q01)
        rows.append({
            "case_id": cid, "rel_dir": rel_dir, "start_idx": st, "variant": "Gaussian",
            **m_g,
        })

    rec01 = run_vultimate_mayo(q01, t=T_INFER, restore_batch=RESTORE_BATCH)
    m_u = vol_metrics(rec01, f01, q01)
    rows.append({
        "case_id": cid, "rel_dir": rel_dir, "start_idx": st, "variant": "V-Ultimate",
        **m_u,
    })

    line = f"  Quarter PSNR={m_q['psnr']:.2f}"
    if MAYO_USE_GAUSSIAN:
        line += f" | Gaussian={m_g['psnr']:.2f}"
    line += f" | V-Ultimate={m_u['psnr']:.2f}"
    print(line)

    del q_hu, f_hu, q01, f01, rec01
    if MAYO_USE_GAUSSIAN:
        del g01
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_raw = pd.DataFrame(rows)
df_raw.to_csv(OUTDIR / "mayo_generalization_raw.csv", index=False)

df_sum = (
    df_raw.groupby("variant", as_index=False)
    .agg(
        n_case=("case_id", "nunique"),
        psnr_mean=("psnr", "mean"),
        psnr_std=("psnr", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        ssim_mean=("ssim", "mean"),
        ssim_std=("ssim", "std"),
        mean_abs_change_vs_quarter=("mean_abs_change_vs_quarter", "mean"),
        max_abs_change_vs_quarter=("max_abs_change_vs_quarter", "mean"),
    )
    .sort_values("psnr_mean", ascending=False)
    .reset_index(drop=True)
)
df_sum.to_csv(OUTDIR / "mayo_generalization_summary.csv", index=False)

print("\n=== Mayo Generalization Summary ===")
display(df_sum)

# ------------------------------------------------------------
# 4) Paired comparison + bootstrap CI
# ------------------------------------------------------------
q = df_raw[df_raw["variant"] == "Quarter"][["case_id", "psnr", "mae", "ssim"]].rename(
    columns={"psnr": "psnr_q", "mae": "mae_q", "ssim": "ssim_q"}
)

paired_rows = []
boot_rows = []

for v in sorted(df_raw["variant"].unique()):
    if v == "Quarter":
        continue

    d = df_raw[df_raw["variant"] == v][["case_id", "psnr", "mae", "ssim"]].rename(
        columns={"psnr": "psnr_v", "mae": "mae_v", "ssim": "ssim_v"}
    )
    m = q.merge(d, on="case_id", how="inner").sort_values("case_id").reset_index(drop=True)

    dpsnr = m["psnr_v"].values - m["psnr_q"].values
    dmae = m["mae_q"].values - m["mae_v"].values
    dssim = m["ssim_v"].values - m["ssim_q"].values

    paired_rows.append({
        "variant": v,
        "n_case": len(m),
        "ΔPSNR_vs_Quarter": float(np.mean(dpsnr)),
        "ΔMAE_vs_Quarter": float(np.mean(dmae)),
        "ΔSSIM_vs_Quarter": float(np.mean(dssim)),
        "PSNR_win_rate": float((m["psnr_v"] > m["psnr_q"]).mean()),
        "MAE_win_rate": float((m["mae_v"] < m["mae_q"]).mean()),
        "SSIM_win_rate": float((m["ssim_v"] > m["ssim_q"]).mean()),
    })

    psnr_mean, psnr_lo, psnr_hi = paired_bootstrap_ci(
        m["psnr_v"].values, m["psnr_q"].values, n_boot=BOOT_N, seed=SEED_MAYO + 1
    )
    mae_mean, mae_lo, mae_hi = paired_bootstrap_ci(
        m["mae_q"].values, m["mae_v"].values, n_boot=BOOT_N, seed=SEED_MAYO + 2
    )
    ssim_mean, ssim_lo, ssim_hi = paired_bootstrap_ci(
        m["ssim_v"].values, m["ssim_q"].values, n_boot=BOOT_N, seed=SEED_MAYO + 3
    )

    boot_rows.extend([
        {"variant": v, "metric": "ΔPSNR_vs_Quarter", "mean": psnr_mean, "ci95_lo": psnr_lo, "ci95_hi": psnr_hi},
        {"variant": v, "metric": "ΔMAE_vs_Quarter",  "mean": mae_mean,  "ci95_lo": mae_lo,  "ci95_hi": mae_hi},
        {"variant": v, "metric": "ΔSSIM_vs_Quarter", "mean": ssim_mean, "ci95_lo": ssim_lo, "ci95_hi": ssim_hi},
    ])

df_pair = pd.DataFrame(paired_rows).sort_values("ΔPSNR_vs_Quarter", ascending=False).reset_index(drop=True)
df_pair.to_csv(OUTDIR / "mayo_generalization_paired_vs_quarter.csv", index=False)

df_boot = pd.DataFrame(boot_rows)
df_boot.to_csv(OUTDIR / "mayo_generalization_bootstrap_vs_quarter.csv", index=False)

print("\n=== Paired vs Quarter ===")
display(df_pair)

print("\n=== Bootstrap CI vs Quarter ===")
display(df_boot)

print("\nSaved:")
for p in sorted(OUTDIR.glob("*.csv")):
    print(" -", p)

## Results Interpretation — Mayo External Paired Evaluation

### 1. What does this experiment test?

This experiment is the key **sim-to-real transfer test** in the project.

The model was trained using **synthetic supervision**, but here it is evaluated on **real external paired low-dose CT data** from the Mayo dataset. Because each quarter-dose volume has a matched full-dose reference, this experiment can directly measure whether the restoration learned from synthetic data transfers to real clinical acquisition.

So the core question here is:

> **Can a restoration model trained with physics-based synthetic supervision improve real external low-dose CT, rather than only working on synthetic test data?**

---

### 2. Dataset construction is strict and conservative

This evaluation uses:

- **10 unique matched Mayo series**
- **strict Quarter/Full pairing by relative directory**
- **one center window per series**
- **no repeated windows from the same series**

That matters because it prevents inflating the sample size by extracting many overlapping subvolumes from the same patient or series.  
In other words, this is a **small but clean external paired evaluation**, not a repeated-window benchmark.

Another notable detail is that the 10 cases are split across:

- **Sharp kernel (D45)**
- **Soft kernel (B30)**

This is useful because it introduces modest reconstruction-domain diversity within the external benchmark.

---

### 3. Main result: V-Ultimate clearly improves real Mayo quarter-dose CT

Compared with the original quarter-dose input:

- **Quarter mean PSNR = 39.49 dB**
- **V-Ultimate mean PSNR = 41.84 dB**
- **ΔPSNR = +2.35 dB**
- **ΔMAE = +0.00203**
- **ΔSSIM = +0.04099**
- **PSNR / MAE / SSIM win rate = 100% / 100% / 100%**

This is the strongest single conclusion from the Mayo experiment:

> **A model trained using synthetic supervision produces consistent, measurable improvement on every paired Mayo case in PSNR, MAE, and SSIM.**

That is a very strong transfer result, especially because this benchmark is external to the training pipeline.

---

### 4. V-Ultimate also outperforms Gaussian in mean PSNR, but that is not the main point

Gaussian is a strong classical baseline here:

- **Gaussian mean PSNR = 41.18 dB**
- **ΔPSNR vs Quarter = +1.69 dB**
- **MAE and SSIM also improve substantially**
- **PSNR win rate vs Quarter = 90%**

So Gaussian is clearly useful. This is important, because it shows that the external Mayo benchmark is not trivial: simple smoothing already helps.

However, V-Ultimate still achieves:

- **higher mean PSNR than Gaussian**
- **larger PSNR gain over Quarter**
- **perfect 10/10 PSNR win rate over Quarter**
- **lower mean MAE than Gaussian**

So the result is not merely that “deep learning beats a weak baseline.”  
Rather, the more precise statement is:

> **Even against a strong and relevant smoothing baseline, V-Ultimate achieves the best overall quantitative fidelity on real external low-dose CT.**

That said, the difference between V-Ultimate and Gaussian should still be interpreted carefully, because the real conceptual contribution here is not just superiority over Gaussian, but successful **external transfer from synthetic supervision to real paired data**.

---

### 5. The most important qualitative finding: V-Ultimate is more conservative than Gaussian

A particularly important result appears in the change-versus-input statistics:

- **Gaussian mean absolute change vs Quarter = 0.00690**
- **V-Ultimate mean absolute change vs Quarter = 0.00316**

and

- **Gaussian max absolute change vs Quarter = 0.15326**
- **V-Ultimate max absolute change vs Quarter = 0.02463**

This is one of the most important findings in the entire external evaluation.

It means that:

> **V-Ultimate achieves higher mean PSNR than Gaussian while making much smaller modifications to the original quarter-dose image.**

In other words, the model is not simply “winning by editing more aggressively.”  
It is actually:

- changing the image **less** on average,
- changing the image **far less** in its worst-case magnitude,
- and still obtaining **better reconstruction fidelity**.

This strongly supports the project’s **controlled restoration / do-no-harm** framing.

A useful way to summarize this is:

> **Gaussian is a stronger global smoother, but V-Ultimate is a more selective and conservative restorer.**

---

### 6. The gain is especially strong in soft-kernel cases, while sharp-kernel cases are more mixed

At the case level, a pattern is visible:

#### Soft kernel (B30) cases
V-Ultimate consistently produces large gains, for example:

- L096: **43.68 → 45.69**
- L333: **42.88 → 44.63**
- L192: **44.20 → 45.77**
- L143: **42.32 → 44.46**

These are strong improvements over both Quarter and Gaussian.

#### Sharp kernel (D45) cases
The picture is more mixed:

- in some sharp-kernel cases, V-Ultimate clearly improves over Quarter and slightly exceeds Gaussian;
- in others, Gaussian is very close or slightly better.

This is a meaningful pattern rather than a weakness to hide.

A reasonable interpretation is:

> **The model transfers particularly well to the smoother soft-kernel reconstruction regime, while the sharper reconstruction regime remains somewhat harder and more variable.**

This is plausible, because sharp-kernel reconstruction tends to preserve more high-frequency texture and noise simultaneously, which makes restoration less straightforward.

---

### 7. Bootstrap confidence intervals confirm the gains are statistically stable

The bootstrap results strengthen the conclusion:

#### V-Ultimate vs Quarter
- **ΔPSNR = +2.35 dB**
- **95% CI: [2.03, 2.67]**
- **ΔMAE = +0.00203**
- **95% CI: [0.00138, 0.00276]**
- **ΔSSIM = +0.04099**
- **95% CI: [0.02542, 0.05881]**

#### Gaussian vs Quarter
- **ΔPSNR = +1.69 dB**
- **95% CI: [0.86, 2.54]**

The key point is that the confidence intervals for V-Ultimate’s improvement over Quarter are **consistently positive**, which supports the conclusion that the transfer effect is not a coincidence driven by a few favorable cases.

So the correct interpretation is:

> **The external Mayo improvement is not only visible at the case level, but also statistically stable under paired bootstrap resampling.**

---

### 8. What does this mean for the overall project claim?

This Mayo experiment is arguably the strongest evidence that the synthetic supervision strategy is genuinely meaningful.

Why? Because the model was **not** trained on real paired Mayo data.  
Yet it still improves real external quarter-dose CT against matched full-dose reference.

That directly supports the central methodological claim:

> **Physics-based synthetic supervision can train a restoration model that transfers to real clinical low-dose CT.**

This is more important than any single comparison against Gaussian.

The main contribution demonstrated here is not simply:

> “Our model has higher PSNR than a classical filter.”

It is:

> **A controlled restoration model trained with synthetic supervision can generalize beyond synthetic data and improve real external paired low-dose CT.**

---

### 9. Final interpretation of this section

The Mayo external paired evaluation supports four key conclusions:

1. **V-Ultimate consistently improves real quarter-dose CT on every evaluated case.**
2. **It achieves the best overall quantitative fidelity among the tested methods.**
3. **It does so while making much smaller changes to the input than Gaussian, which supports the controlled-restoration claim.**
4. **Most importantly, it demonstrates successful sim-to-real transfer from physics-based synthetic supervision to real external clinical data.**

Therefore, the most accurate summary is not just:

> “V-Ultimate beats Gaussian on Mayo.”

It is:

> **V-Ultimate shows that synthetic supervision is not merely a synthetic-domain trick; it can train a restoration model that transfers to real external low-dose CT, improves fidelity, and does so in a more conservative and controlled way than classical smoothing.**

In [ ]:
# Sanity check: verify changed voxel counts under different thresholds
test_case = selected_manifest[0]

q_hu, f_hu, q01, f01 = load_paired_case_window(
    test_case["q_files"], test_case["f_files"], test_case["start_idx"],
    depth=VOL_DEPTH, out_hw=(TARGET_H, TARGET_W)
)
rec01 = run_vultimate_mayo(q01)

change_map = np.abs(rec01 - q01).astype(np.float32)

print("case_id:", test_case["case_id"])
print("mean_abs_change:", float(change_map.mean()))
print("max_abs_change :", float(change_map.max()))

for thr in [0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
    n = int((change_map >= thr).sum())
    print(f"thr={thr:0.3f} -> changed_voxels={n}")

del q_hu, f_hu, q01, f01, rec01, change_map
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# TotalSegmentator on Mayo external paired evaluation (CLEAN / REUSABLE / LOW-PRINT)
# Purpose:
#   1) Reuse the same matched Mayo series from the Mayo cell
#   2) Reuse existing TotalSegmentator outputs when available
#   3) Quantify downstream anatomy benefit (Dice vs Full)
#   4) Quantify anatomical overlap of Quarter->Ultimate edits
#
# Depends on the Mayo cell having already run:
#   - mayo_case_manifest
#   - load_paired_case_window
#   - run_vultimate_mayo
#   - hu01_to_hu
#   - VOL_DEPTH, TARGET_H, TARGET_W
#   - TOTALSEG_OUTDIR
# ============================================================

import os, sys, gc, time, shutil, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import nibabel as nib
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=False)
    import nibabel as nib

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# 0) Checks
# ------------------------------------------------------------
required_globals = [
    "mayo_case_manifest",
    "TOTALSEG_OUTDIR",
    "load_paired_case_window",
    "run_vultimate_mayo",
    "hu01_to_hu",
    "VOL_DEPTH", "TARGET_H", "TARGET_W",
]
missing = [k for k in required_globals if k not in globals()]
if missing:
    raise RuntimeError(f"Run the Mayo cell first. Missing: {missing}")

OUTDIR = Path(TOTALSEG_OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1) Config
# ------------------------------------------------------------
N_TOTALSEG_CASES = min(int(globals().get("N_TOTALSEG_CASES", 10)), len(mayo_case_manifest))
TOTALSEG_CASE_IDS = globals().get("TOTALSEG_CASE_IDS", None)   # optional, e.g. [1,2,3]
TOTALSEG_TASK = str(globals().get("TOTALSEG_TASK", "total"))
TOTALSEG_CHANGE_THR = 0.01
TOTALSEG_FAST = bool(globals().get("TOTALSEG_FAST", True))
TOTALSEG_REUSE_EXISTING = bool(globals().get("TOTALSEG_REUSE_EXISTING", True))
TOTALSEG_FORCE_CPU = bool(globals().get("TOTALSEG_FORCE_CPU", True))
TOTALSEG_SHOW_TOP_K = int(globals().get("TOTALSEG_SHOW_TOP_K", 20))

# ------------------------------------------------------------
# 2) Select cases
# ------------------------------------------------------------
if TOTALSEG_CASE_IDS is not None:
    selected_manifest = [
        m for m in mayo_case_manifest
        if int(m["case_id"]) in set(map(int, TOTALSEG_CASE_IDS))
    ]
else:
    selected_manifest = mayo_case_manifest[:N_TOTALSEG_CASES]

if len(selected_manifest) == 0:
    raise RuntimeError("No Mayo cases selected for TotalSegmentator.")

# ------------------------------------------------------------
# 3) Find / install TotalSegmentator CLI
# ------------------------------------------------------------
cli = None
for c in ["TotalSegmentator", "totalsegmentator"]:
    if shutil.which(c):
        cli = c
        break

if cli is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "TotalSegmentator"], check=False)
    for c in ["TotalSegmentator", "totalsegmentator"]:
        if shutil.which(c):
            cli = c
            break

if cli is None:
    raise RuntimeError("TotalSegmentator CLI not available.")

# ------------------------------------------------------------
# 4) Helpers
# ------------------------------------------------------------
def save_nifti_hu(vol_hu_dhw, out_path):
    affine = np.eye(4, dtype=np.float32)
    vol_hwd = np.transpose(vol_hu_dhw.astype(np.float32), (1, 2, 0))
    nib.save(nib.Nifti1Image(vol_hwd, affine), str(out_path))

def run_totalseg(in_nii, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    existing = list(out_dir.rglob("*.nii.gz"))
    if TOTALSEG_REUSE_EXISTING and len(existing) > 0:
        return 0, "cached"

    cmd = [cli, "-i", str(in_nii), "-o", str(out_dir), "--task", TOTALSEG_TASK]
    if TOTALSEG_FAST:
        cmd.append("--fast")

    env = os.environ.copy()
    if TOTALSEG_FORCE_CPU:
        env["CUDA_VISIBLE_DEVICES"] = ""

    p = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    return p.returncode, p.stdout

def dice_bin(a, b):
    a = (a > 0.5)
    b = (b > 0.5)
    denom = int(a.sum()) + int(b.sum())
    if denom == 0:
        return 1.0
    inter = int((a & b).sum())
    return 2.0 * inter / denom

def resolve_mask(seg_dir, filename):
    p = Path(seg_dir) / filename
    if p.exists():
        return p
    cands = list(Path(seg_dir).rglob(filename))
    return cands[0] if len(cands) else None

# ------------------------------------------------------------
# 5) Main loop
# ------------------------------------------------------------
overlap_rows = []
dice_rows = []
case_rows = []

t0_all = time.time()

for j, case_info in enumerate(selected_manifest, 1):
    cid = int(case_info["case_id"])
    rel_dir = case_info["rel_dir"]
    st = int(case_info["start_idx"])

    print(f"[{j:02d}/{len(selected_manifest)}] case_id={cid:02d}")

    q_hu, f_hu, q01, f01 = load_paired_case_window(
        case_info["q_files"], case_info["f_files"], st,
        depth=VOL_DEPTH, out_hw=(TARGET_H, TARGET_W)
    )
    rec01 = run_vultimate_mayo(q01)

    # Quarter -> Ultimate change map
    change_map = np.abs(rec01 - q01).astype(np.float32)
    change_mask = (change_map >= TOTALSEG_CHANGE_THR).astype(np.uint8)
    total_changed_vox = int(change_mask.sum())

    # Save NIfTI
    safe_rel = rel_dir.replace("/", "__").replace("\\", "__")
    case_dir = OUTDIR / f"case_{cid:02d}_{safe_rel}_st{st:04d}"
    case_dir.mkdir(parents=True, exist_ok=True)

    nii_map = {
        "full": case_dir / "full.nii.gz",
        "quarter": case_dir / "quarter.nii.gz",
        "ultimate": case_dir / "ultimate.nii.gz",
    }
    save_nifti_hu(f_hu, nii_map["full"])
    save_nifti_hu(q_hu, nii_map["quarter"])
    save_nifti_hu(hu01_to_hu(rec01), nii_map["ultimate"])

    # Run / reuse TotalSeg
    seg_dirs = {}
    failed = False
    for nm, nii_p in nii_map.items():
        rc, outtxt = run_totalseg(nii_p, case_dir / f"seg_{nm}")
        if rc != 0:
            failed = True
            seg_dirs[nm] = None
            print(f"   {nm}: FAILED")
            tail = "\n".join(str(outtxt).splitlines()[-8:])
            if tail:
                print(tail)
        else:
            seg_dirs[nm] = case_dir / f"seg_{nm}"

    if failed or seg_dirs["full"] is None or seg_dirs["quarter"] is None or seg_dirs["ultimate"] is None:
        print("   skipped (missing segmentation outputs)")
        del q_hu, f_hu, q01, f01, rec01, change_map, change_mask
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    # overlap + dice
    full_masks = sorted(list(Path(seg_dirs["full"]).rglob("*.nii.gz")))
    change_mask_hwd = np.transpose(change_mask, (1, 2, 0))

    best_mask = None
    best_share = -1.0
    nz_masks = 0

    for mf in full_masks:
        try:
            mask_name = mf.name.replace(".nii.gz", "")
            full_arr = nib.load(str(mf)).get_fdata()
            full_bin = (full_arr > 0.5).astype(np.uint8)
            seg_vox = int(full_bin.sum())
            if seg_vox == 0:
                continue

            inter = int((full_bin * change_mask_hwd).sum())
            if inter > 0:
                nz_masks += 1

            change_in_seg_ratio = (inter / seg_vox) if seg_vox > 0 else np.nan
            seg_share_of_change = (inter / total_changed_vox) if total_changed_vox > 0 else np.nan

            if not np.isnan(seg_share_of_change) and seg_share_of_change > best_share:
                best_share = seg_share_of_change
                best_mask = mask_name

            overlap_rows.append({
                "case_id": cid,
                "rel_dir": rel_dir,
                "start_idx": st,
                "mask_name": mask_name,
                "seg_vox_full": seg_vox,
                "changed_vox_total": total_changed_vox,
                "intersect_change_vox": inter,
                "change_in_seg_ratio": change_in_seg_ratio,
                "seg_share_of_change": seg_share_of_change,
            })

            qmask_path = resolve_mask(seg_dirs["quarter"], mf.name)
            umask_path = resolve_mask(seg_dirs["ultimate"], mf.name)

            if qmask_path is not None and qmask_path.exists():
                q_arr = nib.load(str(qmask_path)).get_fdata()
                dice_rows.append({
                    "case_id": cid,
                    "rel_dir": rel_dir,
                    "start_idx": st,
                    "mask_name": mask_name,
                    "variant": "quarter",
                    "dice_vs_full": float(dice_bin(full_arr, q_arr)),
                })

            if umask_path is not None and umask_path.exists():
                u_arr = nib.load(str(umask_path)).get_fdata()
                dice_rows.append({
                    "case_id": cid,
                    "rel_dir": rel_dir,
                    "start_idx": st,
                    "mask_name": mask_name,
                    "variant": "ultimate",
                    "dice_vs_full": float(dice_bin(full_arr, u_arr)),
                })

        except Exception:
            continue

    case_rows.append({
        "case_id": cid,
        "rel_dir": rel_dir,
        "start_idx": st,
        "changed_vox_total": total_changed_vox,
        "n_masks_total": len(full_masks),
        "n_masks_nonzero": nz_masks,
        "top_mask": best_mask,
        "top_mask_seg_share": best_share if best_share >= 0 else np.nan,
        "mean_abs_change_q_to_u": float(change_map.mean()),
        "max_abs_change_q_to_u": float(change_map.max()),
    })

    del q_hu, f_hu, q01, f01, rec01, change_map, change_mask, change_mask_hwd
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6) Save outputs
# ------------------------------------------------------------
if len(overlap_rows) == 0:
    raise RuntimeError("No TotalSeg overlap rows generated.")

df_case = pd.DataFrame(case_rows)
df_case.to_csv(OUTDIR / "totalseg_case_summary_mayo10.csv", index=False)

df_ov = pd.DataFrame(overlap_rows)
df_ov.to_csv(OUTDIR / "totalseg_overlap_raw_mayo10.csv", index=False)

df_d = pd.DataFrame(dice_rows)
df_d.to_csv(OUTDIR / "totalseg_dice_raw_mayo10.csv", index=False)

ov_sum = (
    df_ov.groupby("mask_name", as_index=False)
    .agg(
        n_case=("case_id", "nunique"),
        n_case_nonzero=("intersect_change_vox", lambda s: int((np.asarray(s) > 0).sum())),
        pct_cases_nonzero=("intersect_change_vox", lambda s: float((np.asarray(s) > 0).mean() * 100.0)),
        mean_change_in_seg_ratio=("change_in_seg_ratio", "mean"),
        median_change_in_seg_ratio=("change_in_seg_ratio", "median"),
        mean_seg_share_of_change=("seg_share_of_change", "mean"),
        median_seg_share_of_change=("seg_share_of_change", "median"),
        max_seg_share_of_change=("seg_share_of_change", "max"),
    )
    .sort_values(["n_case_nonzero", "mean_seg_share_of_change"], ascending=[False, False])
    .reset_index(drop=True)
)
ov_sum.to_csv(OUTDIR / "totalseg_overlap_summary_mayo10.csv", index=False)

if len(df_d) == 0:
    raise RuntimeError("No TotalSeg dice rows generated.")

d_sum = (
    df_d.groupby("variant", as_index=False)
    .agg(
        mean_dice_vs_full=("dice_vs_full", "mean"),
        std_dice_vs_full=("dice_vs_full", "std"),
        median_dice_vs_full=("dice_vs_full", "median"),
        n_rows=("dice_vs_full", "size"),
    )
    .sort_values("mean_dice_vs_full", ascending=False)
    .reset_index(drop=True)
)
d_sum.to_csv(OUTDIR / "totalseg_dice_summary_mayo10.csv", index=False)

qd = df_d[df_d["variant"] == "quarter"][["case_id", "mask_name", "dice_vs_full"]].rename(
    columns={"dice_vs_full": "dice_q"}
)
ud = df_d[df_d["variant"] == "ultimate"][["case_id", "mask_name", "dice_vs_full"]].rename(
    columns={"dice_vs_full": "dice_u"}
)
dm = qd.merge(ud, on=["case_id", "mask_name"], how="inner")
dm["delta_u_minus_q"] = dm["dice_u"] - dm["dice_q"]

d_mask = (
    dm.groupby("mask_name", as_index=False)
    .agg(
        n_case=("case_id", "nunique"),
        mean_delta_u_minus_q=("delta_u_minus_q", "mean"),
        median_delta_u_minus_q=("delta_u_minus_q", "median"),
        pct_cases_improved=("delta_u_minus_q", lambda s: float((np.asarray(s) > 0).mean() * 100.0)),
    )
    .sort_values(["mean_delta_u_minus_q", "pct_cases_improved"], ascending=[False, False])
    .reset_index(drop=True)
)
d_mask.to_csv(OUTDIR / "totalseg_dice_delta_by_mask_mayo10.csv", index=False)

# ------------------------------------------------------------
# 7) Compact display
# ------------------------------------------------------------
elapsed_min = (time.time() - t0_all) / 60.0
print(f"\nDone in {elapsed_min:.1f} min | change_thr={TOTALSEG_CHANGE_THR}")

print("\n=== Case summary ===")
display(df_case)

print("\n=== Dice summary ===")
display(d_sum)

print(f"\n=== Top-{TOTALSEG_SHOW_TOP_K} overlap summary ===")
display(ov_sum.head(TOTALSEG_SHOW_TOP_K))

print(f"\n=== Top-{TOTALSEG_SHOW_TOP_K} dice improvement by mask ===")
display(d_mask.head(TOTALSEG_SHOW_TOP_K))

print("\nSaved:")
for p in sorted(OUTDIR.glob("*mayo10*.csv")):
    print(" -", p)

## Results Interpretation — TotalSegmentator Anatomy Analysis

### 1. What does this experiment test?

This section asks a different question from CRM and Mayo.

CRM shows whether the model can recover clinically relevant signal.  
Mayo shows whether synthetic-supervision training transfers to real external low-dose CT.  
The TotalSegmentator analysis asks:

> **Where is the model actually editing the image, and are those edits anatomically meaningful rather than globally smoothing the whole scan?**

To answer that, we used the restored Mayo volumes and examined:

- **change localization**: where V-Ultimate differs from the original quarter-dose input,
- **anatomical overlap**: which organs and structures those changes fall into,
- **segmentation consistency**: whether the restored image becomes easier to segment relative to the full-dose reference.

This makes TotalSeg an **anatomy-aware interpretability analysis**, not just another image-quality benchmark.

---

### 2. The key result: very small voxelwise edits produce a large segmentation benefit

The most important global result is the Dice summary:

- **Quarter mean Dice vs Full = 0.6137**
- **V-Ultimate mean Dice vs Full = 0.7395**
- **Absolute Dice improvement = +0.1258**

This is a large gain.

At the same time, the per-case intensity change statistics remain modest:

- **mean absolute change (Quarter → Ultimate)** is only about  
  - **0.0015–0.0023** in the soft-kernel cases,
  - **0.0037–0.0065** in the sharp-kernel cases.
- **max absolute change** stays around **0.020–0.026**.

So the strongest first conclusion is:

> **V-Ultimate does not need large image rewrites to create a substantial anatomy-level improvement.**

This is exactly the pattern expected from a controlled restoration model:
small but targeted edits can have a large downstream structural effect.

---

### 3. The edits are not globally uniform; they are anatomically concentrated

The overlap summary shows that changes consistently fall into specific anatomical structures rather than spreading uniformly across the whole scan.

The most frequently involved masks are:

- **small bowel**
- **colon**
- **autochthonous back muscles**
- **liver**
- **heart**
- **iliopsoas**
- **kidney**
- **aorta**
- **vertebrae**

Several of these appear in **100% of applicable cases**, with nonzero overlap between the model’s change map and the segmentation mask.

This supports the interpretation that:

> **The model is not merely denoising the image everywhere equally; it is modifying regions that correspond to real anatomical structures, especially boundary-rich and heterogeneous tissues.**

That is an important qualitative distinction.  
A purely global smoother would be expected to reduce noise broadly, but not necessarily concentrate its effect in anatomically meaningful areas.

---

### 4. Bowel, liver, and muscular structures dominate the change map

The top overlap structures are especially informative:

- **small bowel**
- **colon**
- **autochthon_right**
- **liver**
- **iliopsoas_right**
- **heart**

These are all structures that tend to have:

- complex edges,
- heterogeneous internal texture,
- partial-volume effects,
- and sensitivity to reconstruction noise.

This suggests that V-Ultimate is especially active in regions where low-dose degradation most strongly interferes with anatomical delineation.

A useful interpretation is:

> **The model’s edits cluster in anatomically difficult, boundary-sensitive regions, rather than in homogeneous background or irrelevant areas.**

That is consistent with a restoration mechanism that is doing more than simple smoothing.

---

### 5. Sharp-kernel cases require much more editing than soft-kernel cases

The case summary shows a striking pattern:

- **Sharp-kernel (D45) cases** have very large `changed_vox_total`
- **Soft-kernel (B30) cases** have much smaller `changed_vox_total`

For example:

- Sharp kernel cases: millions of changed voxels
- Soft kernel cases: tens to hundreds of thousands

At the same time:

- **mean absolute change remains small in both groups**
- but is consistently **larger in sharp-kernel cases**

This is a very sensible result.

Sharp-kernel reconstructions preserve more high-frequency structure, but they also amplify noise and edge instability. That means the restoration model needs to intervene more broadly to regularize the image while preserving structure. Soft-kernel cases are already smoother, so fewer corrections are needed.

So this section supports another useful interpretation:

> **The model adapts its editing behavior to reconstruction regime: it edits much more in sharp-kernel data, where the restoration problem is harder, and much less in soft-kernel data, where the input is already more stable.**

That is a sign of conditional, context-sensitive restoration rather than one fixed smoothing behavior.

---

### 6. Liver and bowel are especially important because they account for both overlap frequency and change share

Some structures appear often, but only occupy a tiny fraction of the total change volume.  
Others contribute meaningfully to both:

- how often they are touched,
- and how much of the total change map they account for.

Two especially notable examples are:

- **liver**
- **small bowel / colon**

For instance, the liver reaches very large `seg_share_of_change` values in some cases, especially in the L143 examples, meaning a substantial fraction of the model’s total edits fall inside that organ. Bowel structures also appear consistently across all cases.

This suggests that:

> **The model’s corrections are not only frequent in these structures, but sometimes dominant there, indicating that large portions of the restoration effort are being spent in anatomically meaningful regions.**

This is much more informative than simply saying “the model changes a lot of voxels.”

---

### 7. The strongest segmentation improvements occur in boundary-sensitive and fine structures

The mask-wise Dice improvement table is one of the most revealing parts of the analysis.

The largest gains appear in structures such as:

- **gluteal muscles**
- **vertebrae**
- **ribs**
- **inferior vena cava**
- **pancreas**
- **esophagus**
- **iliopsoas**
- **costal cartilages**
- **portal / splenic veins**

These are structures that are often difficult to segment under low-dose degradation because they depend heavily on:

- clean edge definition,
- preservation of thin boundaries,
- local contrast stability,
- and suppression of noisy texture confusion.

This pattern strongly supports the interpretation that:

> **V-Ultimate is improving anatomical usability at the level of fine structures and boundaries, not merely making images look smoother.**

That is a much stronger claim than a generic denoising statement.

---

### 8. Some structures show high overlap ratio but low total change share — this is not contradictory

A few masks, such as:

- **spinal cord**
- **aorta**
- **vertebrae**

show relatively high `change_in_seg_ratio` but low `seg_share_of_change`.

This is not a contradiction. It simply means:

- a substantial portion of that structure is edited,
- but the structure itself is relatively small compared with the full body volume.

So these results should be interpreted as:

> **The model is highly active within certain compact, clinically meaningful structures, even if those structures do not dominate the total edited voxel count.**

This is actually another sign of targeted behavior.

---

### 9. The threshold matters: `change_thr = 0.01` is permissive, yet the edits still remain structured

This analysis used:

- **change threshold = 0.01**

That is a relatively permissive threshold, meaning even fairly small voxelwise changes are counted as edits. Under such a threshold, a purely indiscriminate method might appear to modify everything.

But even with this permissive setting, the overlap results are still anatomically organized and the downstream Dice improves strongly.

That makes the result more convincing:

> **Even when small edits are counted, the change map still aligns with anatomical structures rather than dissolving into uniform global modification.**

---

### 10. What does this mean for the overall project claim?

This TotalSeg result does not primarily claim that V-Ultimate is “better at segmentation” as an end goal.  
Instead, it provides anatomical evidence for **how** the restoration is behaving.

The overall interpretation is:

1. **The model makes relatively small voxelwise edits;**
2. **those edits are concentrated in real anatomical structures;**
3. **the strongest downstream gains appear in boundary-sensitive tissues and fine structures;**
4. **and the restored image becomes substantially more consistent with the full-dose reference at the segmentation level.**

So the most important conclusion is:

> **V-Ultimate is not acting like a blind global smoother. It is performing anatomically anchored restoration.**

That is exactly the kind of evidence needed to support the broader claim that the model is doing controlled, meaningful recovery rather than arbitrary image manipulation.

---

### 11. Final interpretation of this section

The TotalSegmentator analysis strengthens the project in three major ways:

- **Interpretability:** it shows where the model is editing;
- **Anatomical plausibility:** it shows that edits align with real organs and structures;
- **Functional relevance:** it shows that these edits improve segmentation consistency relative to full-dose CT.

Therefore, the best summary of this section is:

> **V-Ultimate produces small but anatomically structured edits, and those edits translate into a large downstream segmentation improvement. This supports the claim that the model performs controlled, anatomy-aware restoration rather than simple global smoothing.**